In [1]:
!pip install --upgrade numpy
import numpy as np
print(np.__version__)
# NumPy 1.24以降では np.bool8 が削除されたため互換エイリアスを作成
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

2.4.3


In [2]:
!pip install PyGeopack --no-deps
!pip install RecarrayTools DateTimeTools kpindex PyFileIO pyomnidata
!pip install aacgmv2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.1/594.1 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for PyGeopack: filename=PyGeopack-1.2.7-py3-none-any.whl size=2073003 sha256=5537c3240b13749c36a8a681ff146df141d122486212ef0e14a08dbc5e382516
  Stored in directory: /root/.cache/pip/wheels/cf/f3/a6/86d21299677d186115d52a4ce8e8a8805155660bf83775b46b
Successfully built PyGeopack
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 4.6 MB/s eta 0:00:00
  Created wheel for DateTimeTools: filename=DateTimeTools-1.3.0-py3-none-any.whl size=327253 sha256=86103b51063fd23d4cb899ed3ac65d919935f9f3f6773c8f4965c45d961f0340
  Stored in directory: /root/.cache/pip/wheels/62/70/26/25fe0c9b1cb2003a7d8c86de21c9f84031eac168819592c601
Successfully built DateTimeTools
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ..

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
"""
ファイル読み込みから、地理座標変換、地理座標変換法による高度マッピングの関数設定
ファイルをgoogle driveにアップロードする場所（例）
ISSデータ
"/content/drive/MyDrive/Colab Notebooks/Athabaska/data/ISS_data/raw_data/2505_1"
ISSフットプリントにしか使ってないのでCHDの.datのみ
オーロラ画像データ
"/content/drive/MyDrive/Colab Notebooks/Athabaska/data/20250503/AUGSO"
元画像.tiffと強調処理画像.png
"/content/drive/MyDrive/Colab Notebooks/Athabaska/data/20250503/AUGO1"
元画像.tiffのみ
"""
import os
import re
import glob
import csv
import math
from typing import Dict, List, Tuple, Optional
from datetime import datetime, timezone, timedelta

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib import cm

from PIL import Image, ImageOps, ImageDraw, ImageEnhance

import PyGeopack as gp

def read_dat_file(file_path):
    unit = []
    CHD_X = []
    CHD_Y = []
    lat_all = []
    lon_all = []
    h_all = []

    try:
        # load a file with line
        with open(file_path, 'r') as file:
            for line in file:
                # split a line with a blank space to get a value
                columns = line.split()
                if len(columns) < 8:
                    continue

                unit.append(columns[0])
                CHD_X.append(float(columns[2]))
                CHD_Y.append(float(columns[3]))
                lat_all.append(float(columns[4]))
                lon_all.append(float(columns[5]))
                h_all.append(float(columns[6]))

        return unit, CHD_X, CHD_Y, lat_all, lon_all, h_all

    except FileNotFoundError:
        print(f"We couldn't find a file {file_path} ")
        return None
    except ValueError as e:
        print(f"format error of data: {e}")
        return None

def extract_time_from_filename(filepath):
    """
    'YYYY-MM-DD_HHMMSS'
    """
    filename = os.path.basename(filepath)

    match = re.search(r'\d{4}-\d{2}-\d{2}_\d{6}', filename)
    return match.group() if match else None

def timestamp_to_unixtime(timestamp):
    """
    UNIX time
    """
    dt = datetime.strptime(timestamp, '%Y-%m-%d_%H%M%S')

    unixtime = int(dt.timestamp())
    return unixtime
def rotate_and_crop(image, rotation_center, rotation_angle, border, fill=0):
    """
    指定した中心を軸に画像を回転させ、特定の領域を切り取る関数。

    パラメータ: image (PIL.
        image (PIL.Image): 操作対象の画像
        rotation_center (タプル): 回転中心の座標 (x, y)
        rotation_angle (float): 回転角度 (度単位)
        border (int): 拡張する境界線のサイズ
        fill (int): 境界線の色 (デフォルト: 0 = 黒)

    戻り値: PIL.
        PIL.Image: トリミングされた画像
    """
    # Expand an image with a given border
    expanded_image = ImageOps.expand(image, border, fill=fill)

    # Adjust the center of rotation to match the expanded image
    new_center = (rotation_center[0] + border, rotation_center[1] + border)

    # rotate the image at the specified center
    rotated_image = expanded_image.rotate(rotation_angle, center=new_center)

    # rage of trimming
    crop_box = (
        new_center[0] - border,
        new_center[1] - border,
        new_center[0] + border,
        new_center[1] + border
    )

    # trimming image
    cropped_image = rotated_image.crop(crop_box)

    # Get image size after cropping
    xsize, ysize = cropped_image.size
    #print(f'Image size after cropping: {xsize}, {ysize}')

    return cropped_image

def display(image):
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    plt.show()

def rotate_and_crop_enhanced(image, rotation_center, rotation_angle, border, enhance, fill=0):
    if image.mode == '1':
        image = image.convert('L')  # Convert black and white image to grayscale

    expanded_image = ImageOps.expand(image, border, fill=fill)
    new_center = (rotation_center[0] + border, rotation_center[1] + border)
    rotated_image = expanded_image.rotate(rotation_angle, center=new_center)
    crop_box = (
        new_center[0] - border,
        new_center[1] - border,
        new_center[0] + border,
        new_center[1] + border
    )
    cropped_image = rotated_image.crop(crop_box)
    xsize, ysize = cropped_image.size
    #print(f'Image size after cropping: {xsize}, {ysize}')

    #*Histogram stretch**
    np_image = np.array(cropped_image)
    min_val, max_val = np_image.min(), np_image.max()
    if max_val > min_val:
        stretched_image = (np_image - min_val) / (max_val - min_val) * 255
        stretched_image = stretched_image.astype(np.uint8)
        enhanced_image = Image.fromarray(stretched_image)
    else:
        enhanced_image = cropped_image

    #*Advance the contrast**.
    enhancer = ImageEnhance.Contrast(enhanced_image)
    enhanced_image = enhancer.enhance(enhance)


    return enhanced_image


def generate_mapping_table(lat0, lon0, h1, h2, lat_size, lon_size, map_size=888, rr=6371.0):
    """
    緯度・経度マップを生成する関数。

    パラメータ: (float): 基準緯度（ラジアン単位）。
        lat0 (float): 基準緯度（ラジアン単位）
        lon0 (float): 基準経度（ラジアン単位）
        h1 (float): 基準点の高度（km）
        h2 (float): 目標点の高度（km）
        lat_size (float): 緯度範囲（度単位）
        lon_size (float): 経度の範囲（度）
        map_size (int): マッピング画像のサイズ
        rr (float): 地球の半径（km）

    戻り値: xmap, ymap (ピクセル数)
        xmap, ymap (np.ndarray): マッピング用の座標データ

    """
    xmap = np.zeros((map_size, map_size), dtype=float)
    ymap = np.zeros((map_size, map_size), dtype=float)

    for i in range(map_size):
        for j in range(map_size):
            lat = lat0 + lat_size / (map_size / 2) * i * math.radians(1) - lat_size * math.radians(1)
            lon = lon0 + lon_size / (map_size / 2) * j * math.radians(1) - lon_size * math.radians(1)

            b = 0.5 * math.pi - lat
            c = 0.5 * math.pi - lat0
            aa = lon - lon0

            cosa = math.cos(b) * math.cos(c) + math.sin(b) * math.sin(c) * math.cos(aa)
            sina = math.sqrt(1.0 - cosa * cosa)

            if sina == 0:
                continue  # ZeroDivision回避

            sinbb = math.sin(b) * math.sin(aa) / sina
            cosbb = (math.cos(b) * math.sin(c) - math.sin(b) * math.cos(c) * math.cos(aa)) / sina

            the = math.atan2((rr + h2) * sina, (rr + h2) * cosa - (rr + h1))
            l0 = 0.5 * ysize * the / (0.5 * math.pi)

            x = 0.5 * xsize - l0 * sinbb
            y = 0.5 * ysize + l0 * cosbb

            xmap[i, j] = x
            ymap[i, j] = y

    return xmap, ymap

def apply_mapping(image, xmap, ymap, map_size=888):
    """Function to generate a new image using a mapping table."""
    new_image = np.zeros((map_size, map_size), dtype=float)

    for i in range(map_size):
        for j in range(map_size):
            x = int(xmap[i, j])
            y = int(ymap[i, j])

            if 0 <= x < image.width and 0 <= y < image.height:
                new_image[i, j] = image.getpixel((x, y))

    return new_image

def display_image(image, latmin, latmax, lonmin, lonmax, title):
    """Function to display an image on a map."""
    plt.figure()
    plt.imshow(image, extent=(lonmin, lonmax, latmin, latmax), cmap='gray')
    plt.title(f'Mapped Image of {title}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True)
    plt.show()

def save_data(path, **kwargs):
    np.savez(path, **kwargs)

def load_data(path):
    return np.load(path)

def latlon_to_pixel(lat, lon, latmin, latmax, lonmin, lonmax, size):
    i = int((latmax - lat) / (latmax - latmin) * size)
    j = int((lon - lonmin) / (lonmax - lonmin) * size)
    return i, j
def project_latlon_list_to_new_height(latlon_list_100, h_target, lat0, lon0, image_size=888):
    """
    緯度経度リストをlat_center/lon_center基準で放射方向に線形スケールし、
    高度h_targetに対応する緯度経度リストを返す。
    """
    R_earth = 6371.0
    scale = h_target / 100

    latlon_list_scaled = []
    for lat, lon in latlon_list_100:
        delta_lat = lat - lat0
        delta_lon = lon - lon0

        lat_scaled = lat0 + delta_lat * scale
        lon_scaled = lon0 + delta_lon * scale

        latlon_list_scaled.append((lat_scaled, lon_scaled))

    return latlon_list_scaled

def compute_rect_corners(lat1, lat2, lon1, lon2):
    """
    赤枠4隅の緯度経度を計算（左上→左下→右下→右上）
    """
    return [
        (lat2, lon1),  # 左上
        (lat1, lon1),  # 左下
        (lat1, lon2),  # 右下
        (lat2, lon2),  # 右上
    ]

def corners_to_bbox(corners):
    """
    4点 [(lat, lon),...] から (lat_min, lat_max, lon_min, lon_max) のbboxを求める
    """
    lats = [lat for lat, lon in corners]
    lons = [lon for lat, lon in corners]
    lat_min = min(lats)
    lat_max = max(lats)
    lon_min = min(lons)
    lon_max = max(lons)
    return (lat_min, lat_max, lon_min, lon_max)

def get_overlap_bbox(bbox1, bbox2):
    """
    2つの bbox=(lat_min,lat_max,lon_min,lon_max) から共通範囲を計算
    """
    lat_min = max(bbox1[0], bbox2[0])
    lat_max = min(bbox1[1], bbox2[1])
    lon_min = max(bbox1[2], bbox2[2])
    lon_max = min(bbox1[3], bbox2[3])

    if lat_min >= lat_max or lon_min >= lon_max:
        return None  # 重なりなし

    return lat_min, lat_max, lon_min, lon_max

def draw_red_rect_on_map(new_image, image_name, h, rect_latlons, latmin, latmax, lonmin, lonmax):
    """
    赤枠の緯度経度4隅を、地理座標表示(ax)上に赤線で描画する
    """
    # 緯度経度→表示用ピクセル座標
    xs, ys = [], []
    for lat, lon in rect_latlons:
        x = lon
        y = lat
        xs.append(x)
        ys.append(y)

    # 閉じるように先頭点を追加
    xs.append(xs[0])
    ys.append(ys[0])
    fig, ax = plt.subplots()
    ax.imshow(new_image, extent=(lonmin, lonmax, latmin, latmax), cmap='gray')
    ax.plot(xs, ys, color='red', linestyle='-', linewidth=2, label='Red Rect')
    ax.set_title(f"{image_name} {h}km with Red Rect")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.savefig(f"{image_name}_{h}km_with_redrect.png")


# --- 安全な格子インデックス変換 ---
def safe_latlon_to_pixel(lat, lon, latmin, latmax, lonmin, lonmax, map_size_):
    if isinstance(lat, np.ndarray):
        if lat.size == 0:
            return None, None
        lat = lat.item()   # shape=(1,) を想定
    if isinstance(lon, np.ndarray):
        if lon.size == 0:
            return None, None
        lon = lon.item()

    if np.any(np.isnan([lat, lon])): return None, None
    if latmax == latmin or lonmax == lonmin: return None, None
    i = int((latmax - lat) / (latmax - latmin) * map_size_)
    j = int((lon - lonmin) / (lonmax - lonmin) * map_size_)
    if not (0 <= i < map_size_ and 0 <= j < map_size_): return None, None
    return i, j


h1 = 0.0
rr = 6371.0
xsize, ysize = 888, 888
# Central coordinates
lat0_augo1 = 54.71 * math.radians(1)
lon0_augo1 = -113.31 * math.radians(1)
lat0_augso = 54.60 * math.radians(1)
lon0_augso = -113.64 * math.radians(1)

lat_size=3.5
lon_size=5

# Mapping bounds
latmin_augo1, latmax_augo1 = 54.71 - lat_size, 54.71 + lat_size
lonmin_augo1, lonmax_augo1 = -113.31 - lon_size, -113.31 + lon_size
latmin_augso, latmax_augso = 54.60 - lat_size, 54.60 + lat_size
lonmin_augso, lonmax_augso = -113.64 - lon_size, -113.64 + lon_size

map_size = 888

def generate_geomap(h2):
    # Mapping table
    xmap_augo1, ymap_augo1 = generate_mapping_table(lat0_augo1, lon0_augo1, h1, h2, lat_size, lon_size)
    save_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table1.npz', xmap=xmap_augo1, ymap=ymap_augo1)

    xmap_augso, ymap_augso = generate_mapping_table(lat0_augso, lon0_augso, h1, h2, lat_size, lon_size)
    save_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table2.npz', xmap=xmap_augso, ymap=ymap_augso)

    # Load tables
    xmap_augo1 = load_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table1.npz')['xmap']
    ymap_augo1 = load_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table1.npz')['ymap']
    xmap_augso = load_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table2.npz')['xmap']
    ymap_augso = load_data('/content/drive/MyDrive/Colab Notebooks/Athabaska/data/table2.npz')['ymap']


    # Apply mapping
    new_augo1 = apply_mapping(cropped_augo1, xmap_augo1, ymap_augo1)
    new_augso = apply_mapping(cropped_augso, xmap_augso, ymap_augso)
    new_augso2 = apply_mapping(cropped_augso2, xmap_augso, ymap_augso)

    # Save mapped image
    np.save(f'/content/drive/MyDrive/Colab Notebooks/Athabaska/data/20{date}/AUGO1/{time_str}_{h2}_AUGO1', new_augo1)
    np.save(f'/content/drive/MyDrive/Colab Notebooks/Athabaska/data/20{date}/AUGSO/{time_str}_{h2}_AUGSO', new_augso)

    # Get footprint positions
    dt = datetime.fromtimestamp(unixtime, tz=timezone.utc)
    dates = int(dt.strftime("%Y%m%d"))
    ut_time = dt.hour + dt.minute / 60 + dt.second / 3600

    lat_footprints, lon_footprints = [], []

    for lat, lon, alt in zip(lat_list, lon_list, h_list):
        r_geo = (rr + alt) / rr
        lat_rad = np.deg2rad(lat)
        lon_rad = np.deg2rad(lon)

        x_geo = r_geo * np.cos(lat_rad) * np.cos(lon_rad)
        y_geo = r_geo * np.cos(lat_rad) * np.sin(lon_rad)
        z_geo = r_geo * np.sin(lat_rad)

        x_gsm, y_gsm, z_gsm = gp.Coords.GEOtoGSM(x_geo, y_geo, z_geo, dates, ut_time)
        trace = gp.TraceField(x_gsm, y_gsm, z_gsm, dates, ut_time, coord_In='GSM', alt=h2)

        lat_footprints.append(trace.GlatN)
        lon_footprints.append(trace.GlonN)

    print("lat_footprints:", lat_footprints)
    print("lon_footprints:", lon_footprints)

    # ===============================
    #  ここから逆変換：地理座標 → 全天画像ピクセル
    # ===============================

    # AU GO1 用 (全天画像上の x,y)
    xy_points_augo1 = []

    for la, lo in zip(lat_footprints, lon_footprints):
        # lat/lon → マップ格子 (ii, jj)
        ii, jj = safe_latlon_to_pixel(
            la, lo,
            latmin_augo1, latmax_augo1,
            lonmin_augo1, lonmax_augo1,
            xmap_augo1.shape[0]  # = map_size = 888
        )
        if ii is None or jj is None:
            continue
        if not (0 <= ii < xmap_augo1.shape[0] and 0 <= jj < xmap_augo1.shape[1]):
            continue

        # マップ格子 → オリジナル全天画像座標
        x = xmap_augo1[ii, jj]
        y = ymap_augo1[ii, jj]

        # x は左右反転している前提だったので 888 - x
        xy_points_augo1.append((xsize - x, y))

    if len(xy_points_augo1) > 0:
        xy_points_augo1 = np.array(xy_points_augo1)
    else:
        xy_points_augo1 = np.empty((0, 2))

    # AUGSO 用 (全天画像上の x,y)
    xy_points_augso = []
    for la, lo in zip(lat_footprints, lon_footprints):
        ii, jj = safe_latlon_to_pixel(
            la, lo,
            latmin_augso, latmax_augso,
            lonmin_augso, lonmax_augso,
            xmap_augso.shape[0]
        )
        if ii is None or jj is None:
            continue
        if not (0 <= ii < xmap_augso.shape[0] and 0 <= jj < xmap_augso.shape[1]):
            continue

        x = xmap_augso[ii, jj]
        y = ymap_augso[ii, jj]
        xy_points_augso.append((xsize - x, y))

    if len(xy_points_augso) > 0:
        xy_points_augso = np.array(xy_points_augso)
    else:
        xy_points_augso = np.empty((0, 2))

    # Plot AUGO1
    fig, ax = plt.subplots()
    norm1 = mcolors.Normalize(vmin=0, vmax=65504)
    cax = ax.imshow(new_augo1, cmap='gray', extent=(lonmin_augo1, lonmax_augo1, latmin_augo1, latmax_augo1), norm=norm1)
    ax.scatter(lon_footprints, lat_footprints, s=1, color='black')
    cbar = fig.colorbar(cax, ax=ax, fraction=0.03, pad=0.05)
    cbar.set_label('Rayleigh', fontsize=14)
    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x * 0.00458:.2f}'))
    ax.set_title(f'AUGO1 ({h2} km)', fontsize=16)
    ax.set_xlabel('Longitude', fontsize=16)
    ax.set_ylabel('Latitude', fontsize=16)
    ax.grid(True)
    plt.savefig(os.path.join(save_dir, f"{date_time}_geomap_AUGO1.png"), bbox_inches="tight")

    fig, ax = plt.subplots()
    ax.imshow(ImageOps.mirror(cropped_augo1), cmap="gray")
    if xy_points_augo1.size > 0:
        ax.scatter(
            xy_points_augo1[:, 0], xy_points_augo1[:, 1],
            s=6, color='red', alpha=0.7, marker='o', label='Footprint'
        )
        ax.legend(loc='lower right')
    ax.set_title(f'AUGO1 all-sky + footprints ({h2} km)', fontsize=16)
    ax.set_xlim(0, xsize)
    plt.savefig(os.path.join(save_dir, f"{date_time}_allsky_AUGO1.png"),
                bbox_inches="tight")

    # Plot AUGSO
    fig, ax = plt.subplots()
    norm2 = mcolors.Normalize(vmin=0, vmax=65504)
    cax = ax.imshow(new_augso, cmap='gray', extent=(lonmin_augso, lonmax_augso, latmin_augso, latmax_augso), norm=norm2)
    ax.scatter(lon_footprints, lat_footprints, s=1, color='black')
    cbar = fig.colorbar(cax, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label('Rayleigh', fontsize=14)
    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x * 0.00458:.2f}'))
    ax.set_title(f'AUGSO ({h2} km)', fontsize=16)
    ax.set_xlabel('Longitude', fontsize=16)
    ax.set_ylabel('Latitude', fontsize=16)
    ax.grid(True)
    plt.savefig(os.path.join(save_dir, f"{date_time}_geomap_AUGSO.png"), bbox_inches="tight")

    fig, ax = plt.subplots()
    ax.imshow(ImageOps.mirror(cropped_augso), cmap="gray")
    if xy_points_augso.size > 0:
        ax.scatter(
            xy_points_augso[:, 0], xy_points_augso[:, 1],
            s=6, color='red', alpha=0.7, marker='o', label='Footprint'
        )
        ax.legend(loc='lower right')
    ax.set_title(f'AUGSO all-sky + footprints ({h2} km)', fontsize=16)
    ax.set_xlim(0, xsize)
    plt.savefig(os.path.join(save_dir, f"{date_time}_allsky_AUGSO.png"),
                bbox_inches="tight")

    # Plot AUGSO2
    fig, ax = plt.subplots()
    norm2 = mcolors.Normalize(vmin=0, vmax=255)
    cax = ax.imshow(new_augso2, cmap='gray', extent=(lonmin_augso, lonmax_augso, latmin_augso, latmax_augso), norm=norm2)
    ax.scatter(lon_footprints, lat_footprints, s=1, color='red')
    cbar = fig.colorbar(cax, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label('Rayleigh', fontsize=14)
    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x * 0.00458:.2f}'))
    ax.set_title(f'AUGSO ({h2} km)', fontsize=16)
    ax.set_xlabel('Longitude', fontsize=16)
    ax.set_ylabel('Latitude', fontsize=16)
    ax.grid(True)
    plt.savefig(os.path.join(save_dir, f"{date_time}_geomap_AUGSO2.png"), bbox_inches="tight")

    fig, ax = plt.subplots()
    ax.imshow(ImageOps.mirror(cropped_augso2), cmap="gray")
    if xy_points_augso.size > 0:
        ax.scatter(
            xy_points_augso[:, 0], xy_points_augso[:, 1],
            s=6, color='red', alpha=0.7, marker='o', label='Footprint'
        )
        ax.legend(loc='lower right')
    ax.set_title(f'AUGSO all-sky + footprints ({h2} km) enhanced', fontsize=16)
    ax.set_xlim(0, xsize)
    plt.savefig(os.path.join(save_dir, f"{date_time}_allsky_AUGSO2.png"),
                bbox_inches="tight")

    return new_augo1, xmap_augo1, ymap_augo1, new_augso, xmap_augso, ymap_augso, new_augso2, lat_footprints, lon_footprints, xy_points_augo1, xy_points_augso


def _to_date_term(datetime_str: str) -> Tuple[str, str, str]:
    if not re.fullmatch(r"\d{14}", datetime_str):
        raise ValueError("datetime_str must be 14 digits like '20250503093100'")

    yyyy = datetime_str[0:4]
    yy = datetime_str[2:4]
    mm = datetime_str[4:6]
    dd = int(datetime_str[6:8])   # ← int に変換
    hhmmss = datetime_str[8:14]

    # term_month の決定
    if 1 <= dd <= 10:
        term_month = 1
    elif 11 <= dd <= 20:
        term_month = 2
    else:
        term_month = 3

    date = f"{yy}{mm}{dd:02d}"      # '250503'
    term = f"{yy}{mm}_{term_month}" # 例: '2505_1'

    return date, term, hhmmss

def _find_image_file(
    base_dir: str,
    date: str,
    station: str,
    hhmmss: str,
    exts: List[str],
) -> str:
    """
    station: 'AUGO1' or 'AUGSO'
    exts: ['tiff', 'tif', 'png'] etc.
    """
    # ディレクトリ例:
    # /.../Athabaska/data/20250503/AUGO1/
    day_dir = os.path.join(base_dir, f"20{date}", station)

    patterns = []
    for ext in exts:
        patterns.append(os.path.join(day_dir, f"*_{hhmmss}_*.{ext}"))

    candidates: List[str] = []
    for p in patterns:
        candidates.extend(sorted(glob.glob(p)))

    if not candidates:
        raise FileNotFoundError(
            f"No {station} image matching time {hhmmss} in {day_dir}"
        )

    # まずは最初の一致を採用（必要なら選別ロジック拡張可）
    return candidates[0]

def _augso_rotation_params(datetime_str: str) -> Tuple[Tuple[int, int], int]:
    """
    AUGSO の回転パラメータの条件分岐。
    """
    threshold = 20250900000000
    dt_int = int(datetime_str)

    if dt_int >= threshold:
        return (435, 449), 100
    return (421, 460), 10

def load_and_process_athabasca_pair(
    datetime_str: str,
    base_dir: str = "/content/drive/MyDrive/Colab Notebooks/Athabaska/data",
    raw_dir: str = "/content/drive/MyDrive/Colab Notebooks/Athabaska/data/ISS_data/raw_data",
    map_size: int = 888,
    rr: float = 6371.0,
    augo1_rotation_center: Tuple[int, int] = (448, 476),
    augo1_rotation_angle: int = 35,
    border_augo1: int = 444,
    border_augso: int = 444,
    time_window_sec: int = 18,
    augso_prefer_tiff: bool = True,
) -> Dict[str, object]:
    """
    入力時刻に対応する
    - CHD 読み込み
    - 18秒窓の lat/lon/h 抽出
    - AUGO1/AUGSO 画像の自動検索
    - 画像の回転＆クロップ
    を行い、結果を dict で返す。
    """

    date, term, hhmmss = _to_date_term(datetime_str)

    # 1) CHD file
    chd_path = os.path.join(raw_dir, term, f"CHD_{date}.dat")
    unit, CHD_X, CHD_Y, lat_all, lon_all, h_all = read_dat_file(chd_path)

    # 2) 画像ファイルを探索
    # AUGO1 は tiff 優先
    fn_augo1 = _find_image_file(
        base_dir=base_dir,
        date=date,
        station="AUGO1",
        hhmmss=hhmmss,
        exts=["tiff"],
    )

    fn_augso = _find_image_file(
        base_dir=base_dir,
        date=date,
        station="AUGSO",
        hhmmss=hhmmss,
        exts=["tiff"],
    )
    fn_augso2 = _find_image_file(
        base_dir=base_dir,
        date=date,
        station="AUGSO",
        hhmmss=hhmmss,
        exts=["png"],
    )

    # 3) PIL で読み込み
    augo1 = Image.open(fn_augo1)
    augso = Image.open(fn_augso)
    augso2 = Image.open(fn_augso2)

    # 4) unix time を作る
    # ここはあなたの既存関数仕様に合わせて
    # time_str 例: "2025-05-03_093100"
    yyyy = datetime_str[0:4]
    mm = datetime_str[4:6]
    dd = datetime_str[6:8]
    time_str = f"{yyyy}-{mm}-{dd}_{hhmmss}"
    unixtime = timestamp_to_unixtime(time_str)

    # 5) 18秒窓の lat/lon/h
    lat_list: List[float] = []
    lon_list: List[float] = []
    h_list: List[float] = []

    for t, lat, lon, h in zip(unit, lat_all, lon_all, h_all):
        tt = int(float(t))
        if unixtime < tt < unixtime + time_window_sec:
            lat_list.append(lat)
            lon_list.append(lon)
            h_list.append(h)

    # 6) 回転パラメータ
    rotation_center_augso, rotation_angle_augso = _augso_rotation_params(datetime_str)

    # 7) rotate & crop
    cropped_augo1 = rotate_and_crop(
        augo1, augo1_rotation_center, augo1_rotation_angle, border_augo1
    )
    cropped_augso = rotate_and_crop(
        augso, rotation_center_augso, rotation_angle_augso, border_augso
    )
    cropped_augso2 = rotate_and_crop(
        augso2, rotation_center_augso, rotation_angle_augso, border_augso
    )

    save_dir = f"/content/drive/MyDrive/Colab Notebooks/Athabaska/data/20{date}/{hhmmss}"
    os.makedirs(save_dir, exist_ok=True)

    # 8) 返却
    return {
        "datetime_str": datetime_str,
        "date": date,
        "term": term,
        "hhmmss": hhmmss,
        "time_str": time_str,
        "unixtime": unixtime,
        "chd_path": chd_path,
        "fn_augo1": fn_augo1,
        "fn_augso": fn_augso,
        "fn_augso2": fn_augso2,
        "CHD_X": CHD_X,
        "CHD_Y": CHD_Y,
        "lat_list": lat_list,
        "lon_list": lon_list,
        "h_list": h_list,
        "cropped_augo1": cropped_augo1,
        "cropped_augso": cropped_augso,
        "cropped_augso2": cropped_augso2,
        "rotation_center_augo1": augo1_rotation_center,
        "rotation_angle_augo1": augo1_rotation_angle,
        "rotation_center_augso": rotation_center_augso,
        "rotation_angle_augso": rotation_angle_augso,
        "save_dir": save_dir
    }


def _export_dict_to_globals(d):
    for k, v in d.items():
        globals()[k] = v

def bbox_from_rect(latlon_list):
    lats = np.array([p[0] for p in latlon_list])
    lons = np.array([p[1] for p in latlon_list])
    return float(lats.min()), float(lats.max()), float(lons.min()), float(lons.max())

def intersect_bbox(b1, b2):
    """bbox=(latmin,latmax,lonmin,lonmax) の交差。無ければ None"""
    a1,a2,a3,a4 = b1
    b1_,b2_,b3_,b4_ = b2
    latmin = max(a1, b1_)
    latmax = min(a2, b2_)
    lonmin = max(a3, b3_)
    lonmax = min(a4, b4_)
    if (latmax - latmin) <= 0 or (lonmax - lonmin) <= 0:
        return None
    return (latmin, latmax, lonmin, lonmax)

def make_geo_box_edges(lat1, lat2, lon1, lon2, n=300):
    """地理ボックス(緯度lat1–lat2, 経度lon1–lon2)の4辺を返す。"""
    lons = np.linspace(lon1, lon2, n)
    lats = np.linspace(lat1, lat2, n)
    edges = [
        (np.full(n, lat1), lons),  # 下辺
        (np.full(n, lat2), lons),  # 上辺
        (lats, np.full(n, lon1)),  # 左辺
        (lats, np.full(n, lon2)),  # 右辺
    ]
    return edges


def plot_geo_polyline_on_allsky(ax, lat_line, lon_line,
                                xmap, ymap, base_img,
                                latmin, latmax, lonmin, lonmax,
                                color='magenta', lw=1.8, alpha=0.9, label=None):
    """lat/lonの折れ線を全天画像座標に写して描画。"""
    xy = []
    for la, lo in zip(lat_line, lon_line):
        ii, jj = safe_latlon_to_pixel(la, lo, latmin, latmax, lonmin, lonmax, base_img.size[0])
        if ii is None or jj is None:
            continue
        if not (0 <= ii < xmap.shape[0] and 0 <= jj < xmap.shape[1]):
            continue
        x = xmap[ii, jj]
        y = ymap[ii, jj]
        xy.append((888 - x, y))   # ImageOps.mirror と整合するためX反転

    if len(xy) > 1:
        xy = np.array(xy)
        ax.plot(xy[:, 0], xy[:, 1],
                color=color, lw=lw, alpha=alpha, label=label)

def calculate_correlation(lat_center, lon_center, dlat, dlon, corr_thr, frac, width_thr, new_augo1_cache, new_augso_cache):
    max_correlation, optimal_h2 = 0.0, None

    img_bbox_augso = (latmin_augso, latmax_augso, lonmin_augso, lonmax_augso)
    img_bbox_augo1 = (latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1)

    # 追加：相関カーブを保存
    h2_list, corr_list = [], []

    for h2, new_augo1 in new_augo1_cache.items():
        new_augso = new_augso_cache[h2]

        # --- 1) 元矩形(100km箱)→h2 投影 ---
        latlon_list_100 = compute_rect_corners(lat_center - dlat, lat_center + dlat,
                                               lon_center - dlon, lon_center + dlon)
        latlon_list_augso = project_latlon_list_to_new_height(
            latlon_list_100, h_target=h2,
            lat0=np.degrees(lat0_augso), lon0=np.degrees(lon0_augso)
        )
        latlon_list_augo1 = project_latlon_list_to_new_height(
            latlon_list_100, h_target=h2,
            lat0=np.degrees(lat0_augo1), lon0=np.degrees(lon0_augo1)
        )

        # --- 2) 各画像内の交差 → 3) 共通地理範囲 ---
        rect_augso = bbox_from_rect(latlon_list_augso)
        rect_augo1 = bbox_from_rect(latlon_list_augo1)
        inter_augso  = intersect_bbox(rect_augso,  img_bbox_augso)
        inter_augo1  = intersect_bbox(rect_augo1,  img_bbox_augo1)
        if inter_augso is None or inter_augo1 is None:
            continue
        common_geo = intersect_bbox(inter_augso, inter_augo1)
        if common_geo is None:
            continue
        latmin_c, latmax_c, lonmin_c, lonmax_c = common_geo

        # --- 4) 共通 bbox → ピクセル ---
        i1_min, j1_min = latlon_to_pixel(latmax_c, lonmin_c, *img_bbox_augo1, map_size)
        i1_max, j1_max = latlon_to_pixel(latmin_c, lonmax_c, *img_bbox_augo1, map_size)
        i2_min, j2_min = latlon_to_pixel(latmax_c, lonmin_c, *img_bbox_augso, map_size)
        i2_max, j2_max = latlon_to_pixel(latmin_c, lonmax_c, *img_bbox_augso, map_size)
        if None in (i1_min,j1_min,i1_max,j1_max,i2_min,j2_min,i2_max,j2_max):
            continue
        if (i1_max <= i1_min) or (j1_max <= j1_min) or (i2_max <= i2_min) or (j2_max <= j2_min):
            continue

        # --- 5) 切り出し（共通部分のみ） ---
        sub1 = new_augo1[i1_min:i1_max, j1_min:j1_max]
        sub2 = new_augso [i2_min:i2_max, j2_min:j2_max]
        if sub1.size == 0 or sub2.size == 0:
            continue
        r = min(sub1.shape[0], sub2.shape[0]); c = min(sub1.shape[1], sub2.shape[1])
        if r < 3 or c < 3:
            continue
        sub1 = sub1[:r, :c]; sub2 = sub2[:r, :c]

        # --- 6) 相関 ---
        corr = np.corrcoef(sub1.ravel(), sub2.ravel())[0, 1]
        if np.isnan(corr):
            continue

        # カーブ用に保存
        h2_list.append(h2)
        corr_list.append(corr)

        # 従来の最大値更新（暫定）
        if corr > corr_thr and corr > max_correlation:
            max_correlation, optimal_h2 = corr, h2

    # ===== ここから「ピーク明瞭度」チェック =====
    if len(h2_list) == 0:
        return None, None

    h2_arr   = np.array(h2_list)
    corr_arr = np.array(corr_list)

    # 最大
    idx_peak = int(np.argmax(corr_arr))
    h_peak   = h2_arr[idx_peak]
    c_peak   = corr_arr[idx_peak]

    # 1) コントラスト（左右の最大との落差）
    left_max  = np.max(corr_arr[:idx_peak]) if idx_peak > 0 else -np.inf
    right_max = np.max(corr_arr[idx_peak+1:]) if idx_peak+1 < len(corr_arr) else -np.inf
    contrast  = c_peak - max(left_max, right_max)  # 例: > 0.08

    # 2) 値幅（95%幅を推奨）→ シャープさ
    mask_hm = corr_arr >= (frac * c_peak)
    if np.any(mask_hm):
        width_km = h2_arr[mask_hm][-1] - h2_arr[mask_hm][0]  # 例: < 40 km
    else:
        width_km = np.inf

    # 3) 局所ピーク数（多峰性の抑制）
    loc_peaks = 0
    for i in range(1, len(corr_arr)-1):
        if corr_arr[i] > corr_arr[i-1] and corr_arr[i] >= corr_arr[i+1] and corr_arr[i] > (corr_thr - 0.05):
            loc_peaks += 1

    # 4) 二階微分の絶対値（曲率）で尖り具合を補助評価（オプション）
    #    簡易な中央差分
    if 1 <= idx_peak <= len(corr_arr)-2:
        h_step = np.median(np.diff(h2_arr)) if len(h2_arr) > 1 else 10.0
        second_deriv = (corr_arr[idx_peak+1] - 2*c_peak + corr_arr[idx_peak-1]) / (h_step**2)
        curvature = abs(second_deriv)  # 例: > 1e-3
    else:
        curvature = 0.0

    # ===== 判定（必要なら閾値は調整してOK） =====
    # ・相関の高さ
    # ・コントラスト（谷があること）
    # ・半値幅が狭い（シャープ）
    # ・多峰性でない（ピーク数 1）
    # ・曲率が十分（オプション）
    #print(f"{lat_center}, {lon_center},{h_peak},{c_peak},{width_km}")
    cond = ( (c_peak >= corr_thr) and (width_km <= width_thr) )

    if cond:
        return h_peak, c_peak
    else:
        return None, None

def plot_optimal_altitude_map(
    lat1, lat2, dlat, lon1, lon2, dlon,
    hmin, hmax, corr_thr,
    frac, width_thr, alt_for_mlt=100,
    target_mlt_levels=None,
    draw_on='augso', lat_iss1=None, lat_iss2=None, lon_iss3=None, lon_iss4=None
):
    import aacgmv2  # ← 関数内importでスコープ問題を回避
    import numpy as np

    # ========= 前提：あなたの環境に既に存在する変数/関数 =========
    # - cropped_augso / cropped_augo1 : PIL.Image (888x888)
    # - xmap_augso, ymap_augso, xmap_augo1, ymap_augo1 : (888,888) ndarray
    # - latmin_augso, latmax_augso, lonmin_augso, lonmax_augso, map_size
    # - latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1
    # - time_str = 'YYYY-mm-dd_HHMMSS'
    # - apply_mapping, generate_mapping_table, project_latlon_list_to_new_height
    #   corners_to_bbox, get_overlap_bbox, latlon_to_pixel
    # ===============================================================

    # --- 基本設定 ---
    date = datetime.strptime(time_str, "%Y-%m-%d_%H%M%S")
    cmap = cm.jet
    norm = mcolors.Normalize(vmin=hmin, vmax=hmax)

    # --- 走査中心と高度レンジ（相関用） ---
    lat_center_values = np.arange(lat1, lat2, 0.5)
    lon_center_values = np.arange(lon1, lon2, 0.5)
    height_range = np.arange(hmin, hmax, 5)
    LATC, LONC = np.meshgrid(lat_center_values, lon_center_values)
    lat_centers = LATC.ravel()
    lon_centers = LONC.ravel()

    # --- 高度ごとの画像キャッシュ（既存関数を利用） ---
    new_augo1_cache = {
        h2: apply_mapping(cropped_augo1, *generate_mapping_table(lat0_augo1, lon0_augo1, h1, h2, lat_size, lon_size))
        for h2 in height_range
    }
    new_augso_cache = {
        h2: apply_mapping(cropped_augso, *generate_mapping_table(lat0_augso, lon0_augso, h1, h2, lat_size, lon_size))
        for h2 in height_range
    }

    # --- 最適高度の算出 ---
    optimal_heights = []
    for lat_c, lon_c in zip(lat_centers, lon_centers):
        h2, corr = calculate_correlation(lat_c, lon_c, dlat, dlon, corr_thr, frac, width_thr, new_augo1_cache, new_augso_cache)
        if h2 is not None:
            optimal_heights.append((lat_c, lon_c, h2, corr))
    print(f"[INFO] Total valid optimal points: {len(optimal_heights)}")

    # ============================================================
    # 1) 全天画像（AUGSO/AUGO1）へ数値（高度）＋ MLT/MLAT 等値線
    # ============================================================
    if draw_on.lower() == 'augso':
        base_img = cropped_augso2; xmap = xmap_augso; ymap = ymap_augso
        latmin, latmax = latmin_augso, latmax_augso
        lonmin, lonmax = lonmin_augso, lonmax_augso
        xy_points = xy_augso
    else:
        base_img = cropped_augo1; xmap = xmap_augo1; ymap = ymap_augo1
        latmin, latmax = latmin_augo1, latmax_augo1
        lonmin, lonmax = lonmin_augo1, lonmax_augo1
        xy_points = xy_augo1


    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(ImageOps.mirror(base_img), cmap="gray")

    """
    # ============================================================
    # [ADD] 3領域の磁力線を重ね描き（AUGSO / AUGO1 両対応）
    # ============================================================

    # 画像サイズ（mirror 対応用）
    img_w = base_img.size[0]   # PIL Image 前提（例: 888）

    # region ごとの色をここで定義（グローバルに依存しない）
    # ※ region_data に color が無い場合の保険
    default_colors = {
        "R1": "red",
        "R2": "green",
        "R3": "deepskyblue",
    }

    for label, data in region_data.items():

        # ---------- 色の決定 ----------
        if "color" in data:
            col = data["color"]
        else:
            col = default_colors.get(label, "white")

        # ---------- 描画する磁力線の選択 ----------
        if draw_on.lower() == "augso":
            lines = data["lines_augso"]
        else:
            lines = data["lines_augo1"]

        if lines is None or len(lines) == 0:
            continue

        # ---------- 磁力線を scatter で描画 ----------
        first = True  # 凡例用（最初の1本だけ label を付ける）
        for (x_arr, y_arr) in lines:

            # 念のため ndarray 化（型事故防止）
            x_arr = np.asarray(x_arr)
            y_arr = np.asarray(y_arr)

            # 範囲外防止（安全策）
            mask = (
                (x_arr >= 0) & (x_arr < img_w) &
                (y_arr >= 0) & (y_arr < img_w)
            )

            if not np.any(mask):
                continue

            ax.scatter(
                img_w - x_arr[mask],   # mirror 表示に対応した x 反転
                y_arr[mask],
                s=2,
                color=col,
                alpha=0.9,
                linewidths=0,
                label=label if first else None
            )

            first = False

    """
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("MLT (h)", fontsize=16, color='red', labelpad=10)
    ax.set_ylabel("MLAT (deg)", fontsize=16, color='blue', labelpad=10)

    # 高度値のオーバーレイ
    for lat, lon, h2, corr in optimal_heights:
        ii, jj = safe_latlon_to_pixel(lat, lon, latmin, latmax, lonmin, lonmax, base_img.size[0])
        if ii is None or jj is None:
            continue
        if not (0 <= ii < xmap.shape[0] and 0 <= jj < xmap.shape[1]):
            continue
        color = cmap(norm(h2)); x = xmap[ii, jj]; y = ymap[ii, jj]
        ax.text(888 - x, y, f"{int(h2)}", color=color, fontsize=12, ha='center', va='center')

    # === 等値線に使うグリッド（draw_on の範囲） ===
    lat_grid = np.linspace(latmin, latmax, 80)
    lon_grid = np.linspace(lonmin, lonmax, 80)
    LON, LAT = np.meshgrid(lon_grid, lat_grid)
    # flatten → 一括変換（TRACE）
    mlat, mlon, _ = aacgmv2.convert_latlon_arr(LAT.ravel(), LON.ravel(), alt_for_mlt, date, method_code='TRACE')
    mlt = aacgmv2.convert_mlt(mlon, date)
    MLAT = mlat.reshape(LAT.shape); MLT = mlt.reshape(LAT.shape)

    mlat, mlon, _ = aacgmv2.convert_latlon_arr(LAT.ravel(), LON.ravel(), alt_for_mlt, date, method_code='TRACE')
    mlt = aacgmv2.convert_mlt(mlon, date)
    MLAT = mlat.reshape(LAT.shape)
    MLT  = mlt.reshape(LAT.shape)
    # === 追加：MLT unwrap（0–24 の不連続を除去）===
    MLT_unwrap = np.mod(MLT + 12.0, 24.0)

    # === MLT（赤）：0.5h実線＋ラベル、中間は点線 ===
    if target_mlt_levels is None:
        target_mlt_levels_unwrap = np.arange(0.0, 24.0, 0.1)

    fig_tmp, ax_tmp = plt.subplots()
    CS_mlt = ax_tmp.contour(LON, LAT, MLT_unwrap, levels=target_mlt_levels_unwrap, linewidths=0)
    contours_mlt = {lev: CS_mlt.allsegs[i] for i, lev in enumerate(CS_mlt.levels)}
    plt.close(fig_tmp)

    solid_mlt_unwrap = np.arange(0.0, 24.0, 0.5)

    for lev, seglist in contours_mlt.items():
        for seg in seglist:
            if len(seg) < 2:
                continue
            lon_line, lat_line = seg[:, 0], seg[:, 1]
            m = (lat_line>=latmin)&(lat_line<=latmax)&(lon_line>=lonmin)&(lon_line<=lonmax)
            lon_line, lat_line = lon_line[m], lat_line[m]
            if len(lat_line) < 2:
                continue
            xy = []
            for la, lo in zip(lat_line, lon_line):
                ii, jj = safe_latlon_to_pixel(la, lo, latmin, latmax, lonmin, lonmax, base_img.size[0])
                if ii is None or jj is None:
                    continue
                if not (0 <= ii < xmap.shape[0] and 0 <= jj < xmap.shape[1]):
                    continue
                x = xmap[ii, jj]; y = ymap[ii, jj]
                xy.append((888 - x, y))
            if len(xy) > 1:
                xy = np.array(xy)
                is_solid = np.any(np.isclose(lev, solid_mlt_unwrap, atol=1e-3))
                ax.plot(xy[:,0], xy[:,1],
                        lw=1.2 if is_solid else 0.8,
                        color='red', alpha=0.7 if is_solid else 0.5,
                        linestyle='-' if is_solid else '--')
                if is_solid:
                    mid = len(xy)//2
                    label_val = np.mod(lev - 12.0, 24.0)
                    ax.text(xy[mid,0], xy[mid,1], f"{label_val:.1f}h",
                            color='red', alpha=0.7, fontsize=10, ha='center', va='center', fontweight='bold')

    # === MLAT（青）：偶数度 実線＋ラベル、他は点線 ===
    mlat_min, mlat_max = np.nanmin(MLAT), np.nanmax(MLAT)
    mlat_levels = np.arange(np.floor(mlat_min), np.ceil(mlat_max) + 1e-4, 1.0)

    fig_tmp2, ax_tmp2 = plt.subplots()
    CS_mlat = ax_tmp2.contour(LON, LAT, MLAT, levels=mlat_levels, linewidths=0)
    contours_mlat = {lev: CS_mlat.allsegs[i] for i, lev in enumerate(CS_mlat.levels)}
    plt.close(fig_tmp2)

    solid_mlat = np.arange(np.floor(mlat_min/2)*2, np.ceil(mlat_max/2)*2 + 1e-4, 2.0)

    for lev, seglist in contours_mlat.items():
        for seg in seglist:
            if len(seg) < 2:
                continue
            lon_line, lat_line = seg[:, 0], seg[:, 1]
            m = (lat_line>=latmin)&(lat_line<=latmax)&(lon_line>=lonmin)&(lon_line<=lonmax)
            lon_line, lat_line = lon_line[m], lat_line[m]
            if len(lat_line) < 2:
                continue
            xy = []
            for la, lo in zip(lat_line, lon_line):
                ii, jj = safe_latlon_to_pixel(la, lo, latmin, latmax, lonmin, lonmax, base_img.size[0])
                if ii is None or jj is None:
                    continue
                if not (0 <= ii < xmap.shape[0] and 0 <= jj < xmap.shape[1]):
                    continue
                x = xmap[ii, jj]; y = ymap[ii, jj]
                xy.append((888 - x, y))
            if len(xy) > 1:
                xy = np.array(xy)
                is_solid = np.any(np.isclose(lev, solid_mlat, atol=1e-3))
                ax.plot(xy[:,0], xy[:,1],
                        lw=1.0 if is_solid else 0.8,
                        color='blue', alpha=0.6 if is_solid else 0.5,
                        linestyle='-' if is_solid else '--')
                if is_solid:
                    mid = len(xy)//2
                    ax.text(xy[mid,0], xy[mid,1], f"{lev:.0f}°",
                            color='blue', alpha=0.6, fontsize=9, ha='center', va='center', fontweight='bold')

    # --- ISS Footprintなどの点群を全天画像上に描画 ---
    ax.scatter(xy_points[:, 0], xy_points[:, 1], s=6, color='red', alpha=0.7, marker='o', label='Footprint')


    # --- ISS 周辺の地理ボックスを紫枠で描画（オプション） ---
    if (lat_iss1 is not None and lat_iss2 is not None and
        lon_iss3 is not None and lon_iss4 is not None):

        edges = make_geo_box_edges(lat_iss1, lat_iss2, lon_iss3, lon_iss4, n=400)

        first = True
        for lat_edge, lon_edge in edges:
            plot_geo_polyline_on_allsky(
                ax, lat_edge, lon_edge,
                xmap, ymap, base_img,
                latmin, latmax, lonmin, lonmax,
                color='magenta', lw=1.8, alpha=0.9,
                label='ISS around region' if first else None
            )
            first = False

    # カラーバー（高度）
    cbar = fig.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.04, pad=0.04)
    cbar.set_label("Altitude (km)", fontsize=16, labelpad=10)
    ax.set_title(f"{time_str} ({draw_on.upper()})", fontsize=20)
    plt.savefig(os.path.join(save_dir, f"{date_time}_altitude_all-sky.png"), bbox_inches="tight")
    plt.show()


    # ============================================================
    # 2) 地理座標画像（AUGSO 範囲）：高度＋MLT/MLAT＋上軸MLT
    # ============================================================
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(new_augso, extent=(lonmin_augso, lonmax_augso, latmin_augso, latmax_augso), cmap="gray")

    # 高度値（地理座標にそのまま）
    for lat, lon, h2, corr in optimal_heights:
        if np.any(np.isnan([lat, lon, h2])):
            continue
        ax.text(lon, lat, f"{int(h2)}", color=cmap(norm(h2)), fontsize=9, ha='center', va='center')

    # === AUGSO 範囲で LON/LAT を再計算（draw_on が AUGO1 でも破綻しないように） ===
    LONg, LATg = np.meshgrid(
        np.linspace(lonmin_augso, lonmax_augso, 80),
        np.linspace(latmin_augso, latmax_augso, 80)
    )
    mlat_g, mlon_g, _ = aacgmv2.convert_latlon_arr(LATg.ravel(), LONg.ravel(), alt_for_mlt, date, method_code='TRACE')
    mlt_g = aacgmv2.convert_mlt(mlon_g, date)
    MLATg = mlat_g.reshape(LATg.shape); MLTg = mlt_g.reshape(LATg.shape)

    # --- MLT 等値線（赤） ---
    if target_mlt_levels is None:
        mn_g, mx_g = np.nanmin(MLTg), np.nanmax(MLTg)
        target_mlt_levels_g = np.arange(np.floor(mn_g*10)/10, np.ceil(mx_g*10)/10 + 1e-4, 0.1)
    else:
        target_mlt_levels_g = target_mlt_levels

    # MLT（赤）
    CSg_mlt = ax.contour(LONg, LATg, MLTg, levels=target_mlt_levels_g, colors='red', linewidths=0.8, linestyles='--', alpha=0.5)
    solid_mlt_g = np.arange(np.floor(np.nanmin(target_mlt_levels_g)*2)/2, np.ceil(np.nanmax(target_mlt_levels_g)*2)/2 + 1e-4, 0.5)
    CSg_mlt_solid = ax.contour(LONg, LATg, MLTg, levels=solid_mlt_g, colors='red', linewidths=1.2, linestyles='-', alpha=0.9)

    # MLAT（青）
    mlat_min_g, mlat_max_g = np.nanmin(MLATg), np.nanmax(MLATg)
    mlat_levels_g = np.arange(np.floor(mlat_min_g), np.ceil(mlat_max_g)+1e-4, 1.0)
    ax.contour(LONg, LATg, MLATg, levels=mlat_levels_g, colors='blue', linewidths=0.8, linestyles='--', alpha=0.5)
    even_levels_g = np.arange(np.floor(mlat_min_g/2)*2, np.ceil(mlat_max_g/2)*2 + 1e-4, 2.0)
    CSg_mlat_solid = ax.contour(LONg, LATg, MLATg, levels=even_levels_g, colors='blue', linewidths=1.2, linestyles='-', alpha=0.9)
    # -------------------------------
    # 交点ラベル（MLTは上端、MLATは右端）
    # -------------------------------

    def _place_labels_on_edge(ax, CS, edge='top', value_fmt=lambda v: f'{v}', color='k', pad=0.06, fontsize=10, weight='bold'):
        """
        CS: contour set（実線のほう）
        edge: 'top'（y=max） or 'right'（x=max）
        pad: 軸から内側にオフセット（軸長比）
        """
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        dx = (x1 - x0); dy = (y1 - y0)

        # 軸に沿って文字がかぶり過ぎるのを避けるための最小距離（軸長比）
        min_sep = 0.03 * (dx if edge == 'top' else dy)
        placed = []

        for lev, segs in zip(CS.levels, CS.allsegs):
            for seg in segs:
                if len(seg) < 2:
                    continue
                # 各線分と枠との交差を探す
                if edge == 'top':
                    Yb = y1
                    for (xA, yA), (xB, yB) in zip(seg[:-1], seg[1:]):
                        # yA <= Yb <= yB か逆を満たすとき交差
                        if (yA - Yb) * (yB - Yb) <= 0 and (yA != yB):
                            t = (Yb - yA) / (yB - yA)  # 線形補間
                            x_cross = xA + t * (xB - xA)
                            # プロット範囲内かつ、既存ラベルと離れているか
                            if x0 <= x_cross <= x1 and all(abs(x_cross - px) > min_sep for px in placed):
                                placed.append(x_cross)
                                ax.text(x_cross, Yb - pad*dy, value_fmt(lev),
                                    color=color, fontsize=fontsize,
                                    ha='center', va='top', fontweight=weight)
                elif edge == 'right':
                    Xb = x1
                    for (xA, yA), (xB, yB) in zip(seg[:-1], seg[1:]):
                        if (xA - Xb) * (xB - Xb) <= 0 and (xA != xB):
                            t = (Xb - xA) / (xB - xA)
                            y_cross = yA + t * (yB - yA)
                            if y0 <= y_cross <= y1 and all(abs(y_cross - py) > min_sep for py in placed):
                                placed.append(y_cross)
                                ax.text(Xb - pad*dx, y_cross, value_fmt(lev),
                                        color=color, fontsize=fontsize,
                                        ha='right', va='center', fontweight=weight)

    # 上端：MLT（例：1.0h, 1.5h, ... だけ）
    _place_labels_on_edge(ax, CSg_mlt_solid, edge='top', value_fmt=lambda v: f'{v:.1f}h', color='red', pad=0.03, fontsize=11)

    # 右端：MLAT（例：偶数度 64, 62, 60, 58, ...）
    _place_labels_on_edge(ax, CSg_mlat_solid, edge='right', value_fmt=lambda v: f'{int(round(v))}°', color='blue', pad=0.03, fontsize=11)

    # カラーバー（高度）
    cbar = fig.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.04, pad=0.08)
    cbar.set_label("Altitude (km)", fontsize=16, labelpad=10)
    cbar.ax.tick_params(labelsize=12, width=1.0, length=5)

    # 枠・体裁
    ax.plot([lon1, lon2, lon2, lon1, lon1], [lat1, lat1, lat2, lat2, lat1], linestyle='--', color='red', linewidth=1)
    ax.scatter(lon_footprints, lat_footprints, s=1, color='red')
    ax.set_title(f"Emission altitude on geo-map({draw_on.upper()})", fontsize=16)
    ax.set_xlabel("Longitude", fontsize=14, labelpad=8)
    ax.set_ylabel("Latitude", fontsize=14, labelpad=8)
    ax.tick_params(axis='both', labelsize=12, width=1.2, length=6)
    ax_top = ax.secondary_xaxis('top')
    ax_top.set_xlabel("Magnetic Local time (h))", fontsize=16)
    ax_top.tick_params(axis='x', which='both', bottom=False, top=False, labeltop=False)
    ax_right = ax.secondary_yaxis('right')
    ax_right.set_ylabel("Magnetic Latitude (°)", fontsize=16)
    ax_right.tick_params(axis='y', which='both', left=False, right=False, labelright=False)

    plt.show()
    print("[INFO] plot_optimal_altitude_map completed successfully.")


def run_event_sequence(event_dt_list, hmin, hmax, corr_thr, frac, width_thr):
    fmt = "%Y%m%d%H%M%S"

    for dt_str in event_dt_list:
        # 文字列でも datetime でも対応可能にしておく
        if isinstance(dt_str, datetime):
            date_time = dt_str.strftime(fmt)
        else:
            date_time = dt_str

        print(f"=== Processing {date_time} ===")

        # 1) Load & preprocess
        result = load_and_process_athabasca_pair(date_time)

        _export_dict_to_globals(result)
        globals()["date_time"] = result["datetime_str"]

        # 2) Geomap generation
        new_augo1, xmap_augo1, ymap_augo1, \
        new_augso, xmap_augso, ymap_augso, \
        new_augso2, lat_footprints, lon_footprints, \
        xy_augo1, xy_augso = generate_geomap(h2=100)

        _export_dict_to_globals({
            "new_augo1": new_augo1,
            "xmap_augo1": xmap_augo1,
            "ymap_augo1": ymap_augo1,
            "new_augso": new_augso,
            "xmap_augso": xmap_augso,
            "ymap_augso": ymap_augso,
            "new_augso2": new_augso2,
            "lat_footprints": lat_footprints,
            "lon_footprints": lon_footprints,
            "xy_augo1": xy_augo1,
            "xy_augso": xy_augso,
        })

        # 3) Optimal altitude map
        plot_optimal_altitude_map(
            lat1=52, lat2=56, dlat=0.5,
            lon1=-116, lon2=-111, dlon=0.5,
            hmin=hmin, hmax=hmax,
            corr_thr=corr_thr,
            frac=frac,
            width_thr=width_thr,
            alt_for_mlt=100,
            target_mlt_levels=None,
            draw_on='augso',
            lat_iss1=None, lat_iss2=None,
            lon_iss3=None, lon_iss4=None
        )

The $GEOPACK_PATH variable has not been set, this module will not function correctly without it
Data file does not exist - to create data file run "PyGeopack.Params.UpdateParameters()"


In [6]:
"""
指定した時間（例：event_times = ["20250503093340"]（2025年5月3日09:33:40UT）のイベントで、
・元画像の地理座標変換
・ISSフットプリント描画
・地理座標変換法による発光高度（最も相関係数の高い高度）のマッピング
地理座標変換法の条件（例）
求める高度範囲　60km~140km hmin=60, hmax=141（デフォルトで5kmおき）
マッピングする緯度経度範囲（デフォルト）  緯度52~56(0.5おき)、経度-116~-111(0.5おき)
（注：変えたければrun_event_sequence内のplot_optimal_altitude_map(lat1=52, lat2=56, dlat=0.5,
            lon1=-116, lon2=-111, dlon=0.5)を変更）
相関係数の閾値　0.70　corr_thr=0.70
シャープさ：95%幅が50km以下　frac=0.95, width_thr=50
run_event_sequence(event_times, hmin=60, hmax=141, corr_thr=0.70, frac=0.95, width_thr=50)
論文：Figure3b
"""

event_times = ["20250503093340"]
run_event_sequence(event_times, hmin=60, hmax=141, corr_thr=0.70, frac=0.95, width_thr=50)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
event_times = ["20250503092600", "20250503092800", "20250503094800"]
run_event_sequence(event_times, hmin=60, hmax=141, corr_thr=0.70, frac=0.95, width_thr=50)


In [7]:
"""
地理座標変換法で特定の緯度経度範囲を切り取って、高度-相関係数のグラフを作成する関数
論文：Figure3a
"""

from matplotlib.patches import Rectangle
from collections import defaultdict
import os

def _corr_vs_altitude_for_region(lat1, lat2, lon1, lon2, h2_range):
    """
    指定範囲 (lat1~lat2, lon1~lon2) の箱を基準に、
    各 h2 ごとに AUGSO/AGO1 の再投影画像の
    【共通部分のみ】で相関を計算して返す。
    return: np.array(h2_values), np.array(correlations), np.array(pixels_used)
    """
    # --- ユーティリティ（この関数内だけで完結） ---
    def bbox_from_rect(latlon_list):
        lats = np.array([p[0] for p in latlon_list])
        lons = np.array([p[1] for p in latlon_list])
        return float(lats.min()), float(lats.max()), float(lons.min()), float(lons.max())

    def intersect_bbox(b1, b2):
        if b1 is None or b2 is None:
            return None
        a1,a2,a3,a4 = b1   # latmin, latmax, lonmin, lonmax
        c1,c2,c3,c4 = b2
        latmin = max(a1, c1); latmax = min(a2, c2)
        lonmin = max(a3, c3); lonmax = min(a4, c4)
        if (latmax - latmin) <= 0 or (lonmax - lonmin) <= 0:
            return None
        return (latmin, latmax, lonmin, lonmax)

    # 画像の地理範囲（外部で定義済みを使用）
    img_bbox_augso = (latmin_augso, latmax_augso, lonmin_augso, lonmax_augso)
    img_bbox_augo1 = (latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1)

    # 基準（100 km）の矩形
    latlon_list_100 = compute_rect_corners(lat1, lat2, lon1, lon2)

    h2_values, correlations, pixels_used = [], [], []

    for h2 in h2_range:
        # --- 1) h2 へ投影（各サイト視点）
        latlon_list_augso = project_latlon_list_to_new_height(
            latlon_list_100, h_target=h2,
            lat0=np.degrees(lat0_augso), lon0=np.degrees(lon0_augso)
        )
        latlon_list_augo1 = project_latlon_list_to_new_height(
            latlon_list_100, h_target=h2,
            lat0=np.degrees(lat0_augo1), lon0=np.degrees(lon0_augo1)
        )
        rect_augso = bbox_from_rect(latlon_list_augso)
        rect_augo1 = bbox_from_rect(latlon_list_augo1)

        # --- 2) 各画像との交差 → さらに共通部分
        inter_augso = intersect_bbox(rect_augso, img_bbox_augso)
        inter_augo1 = intersect_bbox(rect_augo1, img_bbox_augo1)
        common_geo  = intersect_bbox(inter_augso, inter_augo1)
        if common_geo is None:
            continue  # どこにも共通しない

        latmin_c, latmax_c, lonmin_c, lonmax_c = common_geo

        # --- 3) h1 -> h2 のマッピング画像を生成
        xmap_augo1, ymap_augo1 = generate_mapping_table(lat0_augo1, lon0_augo1, h1, h2, lat_size, lon_size)
        new_augo1 = apply_mapping(cropped_augo1, xmap_augo1, ymap_augo1)

        xmap_augso, ymap_augso = generate_mapping_table(lat0_augso, lon0_augso, h1, h2, lat_size, lon_size)
        new_augso = apply_mapping(cropped_augso, xmap_augso, ymap_augso)

        # --- 4) 共通 bbox を各画像のピクセルへ
        # AGO1
        i1_min, j1_min = latlon_to_pixel(latmax_c, lonmin_c, *img_bbox_augo1, map_size)
        i1_max, j1_max = latlon_to_pixel(latmin_c, lonmax_c, *img_bbox_augo1, map_size)
        # AUGSO
        i2_min, j2_min = latlon_to_pixel(latmax_c, lonmin_c, *img_bbox_augso, map_size)
        i2_max, j2_max = latlon_to_pixel(latmin_c, lonmax_c, *img_bbox_augso, map_size)

        # 無効チェック
        if None in (i1_min,j1_min,i1_max,j1_max,i2_min,j2_min,i2_max,j2_max):
            continue
        if (i1_max <= i1_min) or (j1_max <= j1_min) or (i2_max <= i2_min) or (j2_max <= j2_min):
            continue

        # --- 5) 切り出し & サイズ合わせ（共通部分のみ）
        sub_augo1 = new_augo1[i1_min:i1_max, j1_min:j1_max]
        sub_augso = new_augso[i2_min:i2_max, j2_min:j2_max]
        if sub_augo1.size == 0 or sub_augso.size == 0:
            continue

        r = min(sub_augo1.shape[0], sub_augso.shape[0])
        c = min(sub_augo1.shape[1], sub_augso.shape[1])
        if r < 3 or c < 3:   # 面積が小さすぎるのは除外（必要なら閾値調整）
            continue

        sub_augo1 = sub_augo1[:r, :c]
        sub_augso = sub_augso[:r, :c]

        # --- 6) 相関
        corr = np.corrcoef(sub_augo1.ravel(), sub_augso.ravel())[0, 1]
        if np.isnan(corr):
            continue

        h2_values.append(h2)
        correlations.append(corr)
        pixels_used.append(r * c)

    return np.array(h2_values), np.array(correlations), np.array(pixels_used)

def analyze_multi_regions_correlation_vs_altitude(regions, colors, lines, h2_range):
    """
    複数領域について高度ごとの相関を解析し、
    ・h2 vs correlation グラフ
    ・AUGSO画像に矩形重ね図
    を作成する。

    Parameters
    ----------
    regions : list of dict
        [{'lat1':..., 'lat2':..., 'lon1':..., 'lon2':..., 'label':'R1'}, ...]
    colors : list of str
        各regionに割り当てる色（例: ['red','orange','blue',...])
    lines : list of str
        使用する線種（例: ['-','--',':']）。経度などによって順番に使用。
    h2_range : iterable
        高度[km]の範囲 (例: np.arange(60, 140+1, 5))
    """

    # --- R番号順に色を割り当て ---
    for reg, color in zip(regions, colors):
        reg['color'] = color

    # --- 経度に応じて線種を自動選択 ---
    for i, reg in enumerate(regions):
        reg['linestyle'] = lines[i % len(lines)]

    # --- 結果格納用 ---
    region_results = []
    h2_to_values = defaultdict(list)
    h2_to_pixels = defaultdict(list)

    # --- 各範囲の相関計算 ---

    for reg in regions:
        lat1, lat2, lon1, lon2 = reg['lat1'], reg['lat2'], reg['lon1'], reg['lon2']
        color = reg['color']
        ls = reg['linestyle']
        label = reg['label']
        # 相関を計算
        h2_vals, corrs, pxs = _corr_vs_altitude_for_region(lat1, lat2, lon1, lon2, h2_range)

        region_results.append({
            'label': label,
            'color': color,
            'linestyle': ls,
            'h2': h2_vals,
            'corr': corrs,
            'pixels': pxs,
            'lat1': lat1, 'lat2': lat2,
            'lon1': lon1, 'lon2': lon2
        })

        # 平均曲線用に格納
        for h, c, npx in zip(h2_vals, corrs, pxs):
            h2_to_values[h].append(c)
            h2_to_pixels[h].append(npx)

    # --- 平均曲線を計算 ---
    common_h2 = sorted(h2_to_values.keys())
    avg_corr = np.array([np.mean(h2_to_values[h]) for h in common_h2])

    # --- 図の作成 ---
    fig, ax = plt.subplots(figsize=(10, 5))
    plt.subplots_adjust(right=0.8)

    for res in region_results:
        i = int(np.argmax(res['corr']))  # NEW: ピーク位置を取得
        label_txt = (
            f"{res['label']}  (lat {res['lat1']}–{res['lat2']}°, lon {res['lon1']}–{res['lon2']}°)\n"  # NEW 改行追加
            f"Max corr={res['corr'][i]:.3f} at h2={res['h2'][i]} km"                                   # NEW
        )
        ax.plot(res['h2'], res['corr'],
                color=res['color'],
                ls=res.get('linestyle', '-'),
                lw=2.0,
                alpha=0.9,
                label=label_txt)

    # NEW: 平均カーブのピークを計算して凡例に表示
    if len(common_h2) > 0 and np.all(np.isfinite(avg_corr)):                  # NEW
        i_avg = int(np.nanargmax(avg_corr))                                   # NEW
        avg_peak_h2 = common_h2[i_avg]                                        # NEW
        avg_peak_corr = float(avg_corr[i_avg])                                # NEW
        avg_label = f"Average\nMax corr={avg_peak_corr:.3f} at h2={avg_peak_h2} km"  # NEW
    else:                                                                     # NEW
        avg_label = "Average"                                                 # NEW

    ax.plot(common_h2, avg_corr, color='black', lw=2.5, ls='-', label=avg_label)

    # NEW: 平均曲線の範囲をもとに縦軸範囲を自動決定
    avg_min = np.nanmin(avg_corr)
    y_margin = (1.0 - avg_min) * 0.3 if 1.0 > avg_min else 0.1
    ylim_min = max(avg_min - y_margin, -1.0)
    ylim_max = 1.0
    ax.set_ylim(ylim_min, ylim_max)
    print(f"[INFO] Auto y-limit: {ylim_min:.3f}-{ylim_max:.3f}")

    ax.set_xlabel("Altitude [km]", fontsize=14)
    ax.set_ylabel("Correlation Coefficient", fontsize=14)
    ax.set_title("Correlation vs Altitude across multiple regions", fontsize=16)
    ax.grid(True, alpha=0.3)

    # NEW (optional): 平均ピーク位置に縦線
    if 'avg_peak_h2' in locals():
        ax.axvline(avg_peak_h2, color='black', ls=':', lw=1.5, alpha=0.7)

    # 凡例を右側に出す
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, frameon=False)
    save_path = os.path.join(save_dir, f"{time_str}_correlation_multiple_regions.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    # --- AUGSO の枠重ね表示 ---
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(new_augso2, extent=[lonmin_augso, lonmax_augso, latmin_augso, latmax_augso],
              origin='upper', cmap='gray')
    ax.set_xlabel("Longitude [deg]")
    ax.set_ylabel("Latitude [deg]")
    ax.set_title("AUGSO @ 100 km (All regions)")

    # NEW: オフセットの基準量（図全体の1%）
    jitter_lon = 0.01 * (lonmax_augso - lonmin_augso)
    jitter_lat = 0.01 * (latmax_augso - latmin_augso)
    offsets = [(0, 0), (0, 0), (0, 0), (0, 0), (0.5, 0), (0.5, 0), (0.5, 0), (0.5, 0)]

    for k, reg in enumerate(regions):  # FIX: enumerate を使う
        latlon_list = compute_rect_corners(reg['lat1'], reg['lat2'], reg['lon1'], reg['lon2'])
        lats = [p[0] for p in latlon_list]
        lons = [p[1] for p in latlon_list]
        rect_x = min(lons)
        rect_y = min(lats)
        rect_w = max(lons) - rect_x
        rect_h = max(lats) - rect_y

        ox, oy = offsets[k % len(offsets)]
        dx = ox * jitter_lon
        dy = oy * jitter_lat

        ax.add_patch(Rectangle((rect_x + dx, rect_y + dy), rect_w, rect_h, fill=False, lw=1.2, ec=reg['color'], alpha=0.6, zorder=3))
        ax.text(rect_x + dx + 0.02, rect_y + dy + rect_h - 0.02, reg['label'], color=reg['color'], fontsize=10, va='top', ha='left', zorder=4)

save_path = os.path.join(save_dir, f"{time_str}_region.png")
plt.savefig(save_path, bbox_inches="tight")
plt.show()



def analyze_correlation_vs_altitude(lat1, lat2, lon1, lon2, h2_range):
    h2_values = []
    correlations = []
    latlon_list_100 = compute_rect_corners(lat1, lat2, lon1, lon2)

    for h2 in h2_range:
        latlon_list_augso = project_latlon_list_to_new_height(latlon_list_100, h_target=h2, lat0=np.degrees(lat0_augso), lon0=np.degrees(lon0_augso))
        latlon_list_augo1 = project_latlon_list_to_new_height(latlon_list_100, h_target=h2, lat0=np.degrees(lat0_augo1), lon0=np.degrees(lon0_augo1))
        bbox_augso = corners_to_bbox(latlon_list_augso)
        bbox_augo1 = corners_to_bbox(latlon_list_augo1)
        lat1_scaled, lat2_scaled, lon1_scaled, lon2_scaled = get_overlap_bbox(bbox_augso, bbox_augo1)
        if lat1_scaled is None:
            print(f"No overlap for h2={h2}")
            continue
        latlon_list_new = compute_rect_corners(lat1_scaled, lat2_scaled, lon1_scaled, lon2_scaled)

        xmap_augo1, ymap_augo1_ = generate_mapping_table(lat0_augo1, lon0_augo1, h1, h2, lat_size, lon_size)
        new_augo1 = apply_mapping(cropped_augo1, xmap_augo1, ymap_augo1)
        xmap_augso, ymap_augso = generate_mapping_table(lat0_augso, lon0_augso, h1, h2, lat_size, lon_size)
        new_augso = apply_mapping(cropped_augso, xmap_augso, ymap_augso)

        print(latlon_list_new)
        draw_red_rect_on_map(new_augso, "AUGSO", h2, latlon_list_new, latmin_augso, latmax_augso, lonmin_augso, lonmax_augso)
        draw_red_rect_on_map(new_augo1, "AUGO1", h2, latlon_list_new, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1)

        # Pixel coordinates
        i_augo1_min, j_augo1_min = latlon_to_pixel(lat2_scaled, lon1_scaled, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1, map_size)
        i_augo1_max, j_augo1_max = latlon_to_pixel(lat1_scaled, lon2_scaled, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1, map_size)

        i_augso_min, j_augso_min = latlon_to_pixel(lat2_scaled, lon1_scaled, latmin_augso, latmax_augso, lonmin_augso, lonmax_augso, map_size)
        i_augso_max, j_augso_max = latlon_to_pixel(lat1_scaled, lon2_scaled, latmin_augso, latmax_augso, lonmin_augso, lonmax_augso, map_size)

        # Crop and align images
        sub_augo1 = new_augo1[i_augo1_min:i_augo1_max, j_augo1_min:j_augo1_max]
        sub_augso = new_augso[i_augso_min:i_augso_max, j_augso_min:j_augso_max]

        min_rows = min(sub_augo1.shape[0], sub_augso.shape[0])
        min_cols = min(sub_augo1.shape[1], sub_augso.shape[1])
        sub_augo1 = sub_augo1[:min_rows, :min_cols]
        sub_augso = sub_augso[:min_rows, :min_cols]

        if sub_augo1.shape != sub_augso.shape:
            print("Size mismatch at h2 =", h2)
            continue

        correlation = np.corrcoef(sub_augo1.flatten(), sub_augso.flatten())[0, 1]
        h2_values.append(h2)
        correlations.append(correlation)

    num_pixels = min_rows * min_cols
    print(f"Number of pixels used for correlation: {num_pixels}")

    # Fit cubic polynomial
    coeffs = np.polyfit(h2_values, correlations, 3)
    poly_func = np.poly1d(coeffs)
    fit_correlations = poly_func(h2_values)

    # Get optimal h2
    max_h2_index = np.argmax(fit_correlations)
    optimal_h2 = h2_values[max_h2_index]
    max_correlation = fit_correlations[max_h2_index]

    derivative_poly_func = poly_func.deriv()
    dfdh = derivative_poly_func(optimal_h2)

    SE_r = (1 - max_correlation**2) / np.sqrt(num_pixels - 1)
    sigma_h = SE_r / abs(dfdh)

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(h2_values, correlations, label="Correlation", color="blue", marker="o", markersize=2, linestyle="--")
    plt.xlabel("Altitude [km]", fontsize=20)
    plt.ylabel("Correlation Coefficient", fontsize=20)
    plt.title("Altitude-Correlation Profile", fontsize=24)
    plt.grid()
    plt.legend()
    plt.figtext(1.01, 0.5,
                f"Latitude: {lat1}° to {lat2}°\n"
                f"Longitude: {lon1}° to {lon2}°\n"
                f"Emission altitude: {optimal_h2:.1f} ± {sigma_h:.2f} km\n"
                f"Correlation: {max_correlation:.3f}",
                fontsize=16, color="black", ha="left", va="center")

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{date_time}_correlation_{lat1}_{lat2}_{lon1}_{lon2}.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    # Print result summary
    print(f"[Result] Optimal emission altitude: {optimal_h2:.1f} km ± {sigma_h:.2f} km")
    print(f"Maximum correlation: {max_correlation:.3f}")
    for h2, corr in zip(h2_values, correlations):
        print(f"[result] h2={h2}, correlation={corr:.3f}")

    return optimal_h2, sigma_h

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os

def plot_optimal_altitude_map(lat1, lat2, dlat, lon1, lon2, dlon, hmin, hmax, corr_thr):

    lat_center_values = np.arange(lat1, lat2, 0.5)
    lon_center_values = np.arange(lon1, lon2, 0.5)
    height_range = np.arange(hmin, hmax, 10)

    # Color map setup
    cmap = cm.jet
    norm = mcolors.Normalize(vmin=hmin, vmax=hmax)

    lat_centers, lon_centers = np.meshgrid(lat_center_values, lon_center_values)
    lat_centers = lat_centers.flatten()
    lon_centers = lon_centers.flatten()

    # Cache mapped images
    new_augo1_cache = {
        h2: apply_mapping(cropped_augo1, *generate_mapping_table(lat0_augo1, lon0_augo1, h1, h2, lat_size, lon_size))
        for h2 in height_range
    }
    new_augso_cache = {
        h2: apply_mapping(cropped_augso, *generate_mapping_table(lat0_augso, lon0_augso, h1, h2, lat_size, lon_size))
        for h2 in height_range
    }

    def calculate_correlation(lat_center, lon_center):
        max_correlation = 0
        optimal_h2 = None
        for h2, new_augo1 in new_augo1_cache.items():
            new_augso = new_augso_cache[h2]
            latlon_list_100 = compute_rect_corners(lat_center-dlat, lat_center + dlat, lon_center - dlon, lon_center + dlon)
            latlon_list_augso = project_latlon_list_to_new_height(latlon_list_100, h_target=h2, lat0=np.degrees(lat0_augso), lon0=np.degrees(lon0_augso))
            latlon_list_augo1 = project_latlon_list_to_new_height(latlon_list_100, h_target=h2, lat0=np.degrees(lat0_augo1), lon0=np.degrees(lon0_augo1))
            bbox_augso = corners_to_bbox(latlon_list_augso)
            bbox_augo1 = corners_to_bbox(latlon_list_augo1)
            lat1_scaled, lat2_scaled, lon1_scaled, lon2_scaled = get_overlap_bbox(bbox_augso, bbox_augo1)
            if lat1_scaled is None:
                print(f"No overlap for h2={h2}")
                continue
            i1_min, j1_min = latlon_to_pixel(lat2_scaled, lon1_scaled, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1, map_size)
            i1_max, j1_max = latlon_to_pixel(lat1_scaled, lon2_scaled, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1, map_size)
            i2_min, j2_min = latlon_to_pixel(lat2_scaled, lon1_scaled, latmin_augso, latmax_augso, lonmin_augso, lonmax_augso, map_size)
            i2_max, j2_max = latlon_to_pixel(lat1_scaled, lon2_scaled, latmin_augso, latmax_augso, lonmin_augso, lonmax_augso, map_size)

            sub1 = new_augo1[i1_min:i1_max, j1_min:j1_max]
            sub2 = new_augso[i2_min:i2_max, j2_min:j2_max]

            min_rows = min(sub1.shape[0], sub2.shape[0])
            min_cols = min(sub1.shape[1], sub2.shape[1])
            sub1 = sub1[:min_rows, :min_cols]
            sub2 = sub2[:min_rows, :min_cols]

            corr = np.corrcoef(sub1.flatten(), sub2.flatten())[0, 1]
            if corr > corr_thr and corr > max_correlation:
                max_correlation = corr
                optimal_h2 = h2
        return optimal_h2, max_correlation

    # Run calculations
    optimal_heights = []
    for lat_c, lon_c in zip(lat_centers, lon_centers):
        h2, corr = calculate_correlation(lat_c, lon_c)
        if h2 is not None:
            optimal_heights.append((lat_c, lon_c, h2, corr))



    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(cropped_augo1, cmap="gray")  # オーロラ画像を背景として表示

    # optimal_heightsから緯度経度と高度を抽出
    for lat, lon, h2, corr in optimal_heights:
        # 緯度経度→ピクセル座標変換
        i, j = latlon_to_pixel(lat, lon, latmin_augo1, latmax_augo1, lonmin_augo1, lonmax_augo1, cropped_augo1.size[0])
        x = xmap_augo1[i,j]
        y = ymap_augo1[i,j]
        ax.scatter(x, y, c=[[h2]], cmap="jet", vmin=hmin, vmax=hmax, s=1, alpha=0.8)

     # カラーバーを追加
#    cbar = plt.colorbar(cm.ScalarMappable(cmap="jet", norm=mcolors.Normalize(vmin=hmin, vmax=hmax)), ax=ax, label="Emission Altitude (km)")
    lats_plot, lons_plot, colors = zip(*[(lat, lon, h2) for lat, lon, h2, _ in optimal_heights])
    sc = ax.scatter(lons_plot, lats_plot, c=colors, s=1, cmap="jet", norm=norm, alpha=0.8)
    cbar = fig.colorbar(sc, ax=ax, label="Altitude (h2)", fraction=0.03, pad=0.04)

    ax.set_title("Emission Altitude Overlay on Original Aurora Image", fontsize=20)
    ax.set_xlabel("Pixels")
    ax.set_ylabel("Pixels")
    save_path = os.path.join(save_dir, f"{date_time}_mapping.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(new_augso, extent=(lonmin_augso, lonmax_augso, latmin_augso, latmax_augso), cmap="gray")

    lats_plot, lons_plot, colors = zip(*[(lat, lon, h2) for lat, lon, h2, _ in optimal_heights])
    sc = ax.scatter(lons_plot, lats_plot, c=colors, s=0.3, cmap=cmap, norm=norm, alpha=0.9)

    cbar = fig.colorbar(sc, ax=ax, label="Altitude (h2)", fraction=0.03, pad=0.04)
    cbar.set_ticks(np.arange(hmin, hmax + 1, 10))

    ax.plot([lon1, lon2, lon2, lon1, lon1], [lat1, lat1, lat2, lat2, lat1],
            linestyle='--', color='red', linewidth=1)

    ax.set_title("Optimal Heights with Correlation", fontsize=20)
    ax.set_xlabel("Longitude", fontsize=20)
    ax.set_ylabel("Latitude", fontsize=20)

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{time_str}_mapping_{lat1}_{lat2}_{lon1}_{lon2}.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    return

<Figure size 640x480 with 0 Axes>

In [21]:
"""
地理座標変換法で特定の緯度経度範囲を切り取って、高度-相関係数のグラフを作成
例：
求める高度範囲　60km~140km 5kmおき
求める緯度経度範囲 緯度：52~54 経度：-112.5~-111
h2_range = np.arange(60, 141, 5)
optimal_h2_1, sigma_h_1 = analyze_correlation_vs_altitude(lat1=52, lat2=54, lon1=-112.5, lon2=-111, h2_range=h2_range)
論文：Figure3a
"""

h2_range = np.arange(60, 141, 5)
optimal_h2_1, sigma_h_1 = analyze_correlation_vs_altitude(lat1=52, lat2=54, lon1=-112.5, lon2=-111, h2_range=h2_range)

Output hidden; open in https://colab.research.google.com to view.

In [8]:
"""
磁力線トレーシング法の関数設定
"""

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import os
import math
import PyGeopack as gp
from datetime import datetime, timezone
from scipy.signal import medfilt

# UNIX time → UTC
dt = datetime.fromtimestamp(unixtime, tz=timezone.utc)
# Create date in YYYYYMMDD format
dates = int(dt.strftime("%Y%m%d"))

# Calculate UT (time) (decimal format: hours + minutes/60 + seconds/3600)
ut_time = dt.hour + dt.minute / 60 + dt.second / 3600

def field_line_tracing(lat_fl, lon_fl, alt_fl, upper, lower, interval):
    """
    alt_fl を中心高度としつつ，
    実際のトレース開始点は (alt_fl + upper) にとって，
    そこから下方向にだけ磁力線をたどる版。
    """

    lat_fieldline = []
    lon_fieldline = []
    alt_fieldline = []

    # トレースする高度範囲（上→下）
    alt_top    = alt_fl + upper
    alt_bottom = alt_fl - lower
    alt_range  = np.arange(alt_top, alt_bottom - 1e-6, -interval)

    # 開始点は alt_top（＝ alt_fl + upper）
    r_geo = (rr + alt_top) / rr   # Re 単位

    lat_rad = np.deg2rad(lat_fl)
    lon_rad = np.deg2rad(lon_fl)

    # (x, y, z) in GEO at alt_top
    x_geo = r_geo * np.cos(lat_rad) * np.cos(lon_rad)
    y_geo = r_geo * np.cos(lat_rad) * np.sin(lon_rad)
    z_geo = r_geo * np.sin(lat_rad)

    # GEO → GSM （開始点はずっと同じ）
    x_gsm, y_gsm, z_gsm = gp.Coords.GEOtoGSM(x_geo, y_geo, z_geo, dates, ut_time)

    # alt_top から alt_bottom まで，下方向だけをトレース
    for alt in alt_range:
        trace = gp.TraceField(x_gsm, y_gsm, z_gsm,
                              dates, ut_time,
                              coord_In='GSM', alt=alt)

        lat_fieldline.append(trace.GlatN)
        lon_fieldline.append(trace.GlonN)
        alt_fieldline.append(alt)

    # ここで高度を「低→高」の昇順に並べ直しておくと，
    # 既存のプロファイル描画と同じ向きになる
    lat_fieldline  = lat_fieldline[::-1]
    lon_fieldline  = lon_fieldline[::-1]
    alt_fieldline  = alt_fieldline[::-1]

    return lat_fieldline, lon_fieldline, alt_fieldline

cropped_augo1_np = np.array(cropped_augo1, dtype=np.uint16)
cropped_augso_np = np.array(cropped_augso, dtype=np.uint16)

DEBUG = True  # ← デバッグ出力を有効化するトグル

def inverse_mapping(lat_fieldline, lon_fieldline, alt_fieldline, lat0, lon0, h1, lat_size, lon_size, map_size=1024, rr=6371.0, debug_tag=None):
    """
    Function to find pixel coordinates on an image from latitude, longitude, and altitude.
    """
    lat_fieldline_rad = np.radians(lat_fieldline)
    lon_fieldline_rad = np.radians(lon_fieldline)

    x_points = []
    y_points = []

    # DEBUG カウンタ
    dbg_total = 0             # DEBUG
    dbg_sina_zero = 0         # DEBUG
    dbg_inbounds = 0          # DEBUG
    dbg_oob = 0               # DEBUG

    for lat, lon, alt in zip(lat_fieldline_rad, lon_fieldline_rad, alt_fieldline):
        if isinstance(lat, np.ndarray):
            lat = lat.item()
        if isinstance(lon, np.ndarray):
            lon = lon.item()
        if isinstance(alt, np.ndarray):
            alt = alt.item()
        dbg_total += 1  # DEBUG

        b = 0.5 * math.pi - lat
        c = 0.5 * math.pi - lat0
        aa = lon - lon0

        cosa = math.cos(b) * math.cos(c) + math.sin(b) * math.sin(c) * math.cos(aa)
        sina = math.sqrt(1.0 - cosa * cosa)

        if sina == 0:
            dbg_sina_zero += 1  # DEBUG
            continue

        sinbb = math.sin(b) * math.sin(aa) / sina
        cosbb = (math.cos(b) * math.sin(c) - math.sin(b) * math.cos(c) * math.cos(aa)) / sina

        the = math.atan2((rr + alt) * sina, (rr + alt) * cosa - (rr + h1))
        l0 = 0.5 * ysize * the / (0.5 * math.pi)

        x = 0.5 * xsize - l0 * sinbb
        y = 0.5 * ysize - l0 * cosbb

        i = int(round(y))
        j = int(round(x))

        if 0 <= i < map_size and 0 <= j < map_size:
            y_points.append(i)
            x_points.append(j)
            dbg_inbounds += 1  # DEBUG
        else:
            dbg_oob += 1  # DEBUG
            # 既存はスキップ（ここではまだNaN補完しない）

    if DEBUG:  # DEBUG
        tag = f"[{debug_tag}]" if debug_tag else ""  # DEBUG
        #print(f"DEBUG{tag} inverse_mapping: total={dbg_total}, inbounds={dbg_inbounds}, oob={dbg_oob}, sina_zero={dbg_sina_zero}, returned={len(x_points)}")  # DEBUG

    return x_points, y_points


def rayleigh_vs_altitude(x_points_augo1, y_points_augo1, x_points_augso, y_points_augso, alt_fieldline):
    rayleigh_augo1 = []
    rayleigh_augso = []

    for x, y in zip(x_points_augo1, y_points_augo1):
        if 0 <= y < cropped_augo1_np.shape[0] and 0 <= x < cropped_augo1_np.shape[1]:
            rayleigh_augo1.append(cropped_augo1_np[int(y), int(x)] * 0.00458)
        else:
            rayleigh_augo1.append(np.nan)

    for x, y in zip(x_points_augso, y_points_augso):
        if 0 <= y < cropped_augso_np.shape[0] and 0 <= x < cropped_augso_np.shape[1]:
            rayleigh_augso.append(cropped_augso_np[int(y), int(x)] * 0.00458)
        else:
            rayleigh_augso.append(np.nan)

    rayleigh_augo1 = np.array(rayleigh_augo1)
    rayleigh_augso = np.array(rayleigh_augso)
    alt_fieldline = np.array(alt_fieldline)

    if DEBUG:  # DEBUG
        n1 = np.isfinite(rayleigh_augo1).sum()  # DEBUG
        n2 = np.isfinite(rayleigh_augso).sum()  # DEBUG
        #print(f"DEBUG [rayleigh_vs_altitude] finite: AUGO1={n1}/{len(rayleigh_augo1)}, AUGSO={n2}/{len(rayleigh_augso)}, alt_len={len(alt_fieldline)}")  # DEBUG

    return rayleigh_augo1, rayleigh_augso, alt_fieldline

def plot_profiles_with_mean(profiles_augo1, profiles_augso, alt_fieldline, colors, labels, title_augo1="Rayleigh vs Altitude (AUGO1)", title_augso="Rayleigh vs Altitude (AUGSO)", save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt

    arr_augo1 = np.vstack(profiles_augo1) if len(profiles_augo1) > 0 else np.empty((0, len(alt_fieldline)))
    arr_augso = np.vstack(profiles_augso) if len(profiles_augso) > 0 else np.empty((0, len(alt_fieldline)))

    mean_augo1 = np.nanmean(arr_augo1, axis=0) if arr_augo1.size else np.full_like(alt_fieldline, np.nan, dtype=float)
    mean_augso = np.nanmean(arr_augso, axis=0) if arr_augso.size else np.full_like(alt_fieldline, np.nan, dtype=float)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

    # 個別線（AUGO1）
    for prof, c, lab in zip(profiles_augo1, colors, labels):
        ax1.plot(prof, alt_fieldline, '-', lw=1.2, alpha=0.9, color=c, label=lab)
    # 平均（AUGO1）
    ax1.plot(mean_augo1, alt_fieldline, '-', lw=3.0, color='black', label='Mean')

    ax1.set_title(title_augo1, fontsize=16)
    ax1.set_xlabel("Rayleigh", fontsize=14)
    ax1.set_ylabel("Altitude (km)", fontsize=14)
    ax1.grid(True)
    ax1.legend(fontsize=8, loc='best', ncol=1)

    # 個別線（AUGSO）
    for prof, c, lab in zip(profiles_augso, colors, labels):
        ax2.plot(prof, alt_fieldline, '-', lw=1.2, alpha=0.9, color=c, label=lab)
    # 平均（AUGSO）
    ax2.plot(mean_augso, alt_fieldline, '-', lw=3.0, color='black', label='Mean')

    ax2.set_title(title_augso, fontsize=16)
    ax2.set_xlabel("Rayleigh", fontsize=14)
    ax2.grid(True)
    ax2.legend(fontsize=8, loc='best', ncol=1)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Profiles (with mean) saved at: {save_path}")

    plt.show()
    #print(f"DEBUG: len(profiles_augo1)={len(profiles_augo1)}, "f"len(profiles_augso)={len(profiles_augso)}, "f"len(colors)={len(colors)}, len(labels)={len(labels)}")
    #for i, (p1, p2) in enumerate(zip(profiles_augo1, profiles_augso)):
        #print(f"DEBUG profile {i}: len(AUGO1)={len(p1)}, len(AUGSO)={len(p2)}")

    return mean_augo1, mean_augso


# Fit a cubic polynomial (3rd degree) and find the peak (maximum value)
def fit_and_find_peak_med(altitudes, rayleigh_values):
    valid_idx = ~np.isnan(rayleigh_values)
    altitudes, rayleigh_values = altitudes[valid_idx], rayleigh_values[valid_idx]

    if len(altitudes) == 0:
        return None, None, None

    rayleigh_smooth = medfilt(rayleigh_values, kernel_size=3)

    peak_idx = np.argmax(rayleigh_smooth)
    peak_altitude = altitudes[peak_idx]
    peak_rayleigh = rayleigh_smooth[peak_idx]

    return altitudes, rayleigh_smooth, (peak_altitude, peak_rayleigh)

def fit_and_find_peak(altitudes, rayleigh_values):
    valid_idx = ~np.isnan(rayleigh_values)  # Remove NaN values
    altitudes, rayleigh_values = altitudes[valid_idx], rayleigh_values[valid_idx]

    if len(altitudes) < 4:  # Not enough data for cubic fitting
        return None, None, None

    poly_coeffs = np.polyfit(altitudes, rayleigh_values, 3)  # Cubic fit
    poly_func = np.poly1d(poly_coeffs)  # Create polynomial function

    # Generate smooth curve for plotting
    alt_smooth = np.linspace(min(altitudes), max(altitudes), 100)
    rayleigh_smooth = poly_func(alt_smooth)

    # Find peak (maximum value of the fitted curve)
    peak_idx = np.argmax(rayleigh_smooth)
    peak_altitude = alt_smooth[peak_idx]
    peak_rayleigh = rayleigh_smooth[peak_idx]

    return poly_func, peak_altitude, peak_rayleigh
def plot_pixel_values_vs_altitude(rayleigh_augo1, rayleigh_augso, alt_fieldline, title_augo1, title_augso, save_path, save_condition):
    """
    Obtains the image luminance values from the specified pixel coordinates and plots the relationship between elevation and pixel values.
    Additionally, applies cubic polynomial fitting and plots the fitted curves along with peak values.

    Parameters:
        rayleigh_augo1 (list): list of Rayleigh values for AUGO1.
        rayleigh_augso (list): list of Rayleigh values for AUGSO.
        alt_fieldline (list): list of altitudes for each point.
    """

#    poly_func_augo1, peak_alt_augo1, peak_rayleigh_augo1 = fit_and_find_peak(alt_fieldline, rayleigh_augo1)
#    poly_func_augso, peak_alt_augso, peak_rayleigh_augso = fit_and_find_peak(alt_fieldline, rayleigh_augso)

    altitudes_augo1, rayleigh_smooth_augo1, (peak_alt_augo1, peak_rayleigh_augo1) = fit_and_find_peak_med(alt_fieldline, rayleigh_augo1)
    altitudes_augso, rayleigh_smooth_augso, (peak_alt_augso, peak_rayleigh_augso) = fit_and_find_peak_med(alt_fieldline, rayleigh_augso)
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Plot AUGO1
#    ax1.plot(rayleigh_augo1, alt_fieldline, 'ro-', label="cropped_augo1")  # Original data
#    if poly_func_augo1 is not None:
#        alt_smooth = np.linspace(min(alt_fieldline), max(alt_fieldline), 100)
#        ax1.plot(poly_func_augo1(alt_smooth), alt_smooth, 'r--', label="Cubic Fit")  # Fitting curve
#        ax1.plot(peak_rayleigh_augo1, peak_alt_augo1, 'rx', markersize=10, label="Peak")  # Peak point
    if altitudes_augo1 is not None:
        ax1.plot(rayleigh_augo1, alt_fieldline, 'ro-', alpha=0.5, label="Original Data")
        ax1.plot(rayleigh_smooth_augo1, altitudes_augo1, 'r-', label="Median Filtered")
        ax1.plot(peak_rayleigh_augo1, peak_alt_augo1, 'rx', markersize=10, label="Peak")

    ax1.set_ylabel("Altitude (km)", fontsize=20)
    ax1.set_xlabel("Rayleigh", fontsize=20)
    ax1.set_title(title_augo1, fontsize=20)
    ax1.legend()
    ax1.grid(True)

    # Plot AUGSO
#    ax2.plot(rayleigh_augso, alt_fieldline, 'bo-', label="cropped_augso")  # Original data
#    if poly_func_augso is not None:
#        alt_smooth = np.linspace(min(alt_fieldline), max(alt_fieldline), 100)
#        ax2.plot(poly_func_augso(alt_smooth), alt_smooth, 'b--', label="Cubic Fit")  # Fitting curve
#        ax2.plot(peak_rayleigh_augso, peak_alt_augso, 'bx', markersize=10, label="Peak")  # Peak point
    if altitudes_augso is not None:
        ax2.plot(rayleigh_augso, alt_fieldline, 'bo-', alpha=0.5, label="Original Data")
        ax2.plot(rayleigh_smooth_augso, altitudes_augso, 'b-', label="Median Filtered")
        ax2.plot(peak_rayleigh_augso, peak_alt_augso, 'bx', markersize=10, label="Peak")

    ax2.set_ylabel("Altitude (km)", fontsize=20)
    ax2.set_xlabel("Rayleigh", fontsize=20)
    ax2.set_title(title_augso, fontsize=20)
    ax2.legend()
    ax2.grid(True)
    plt.figtext(1.01, 0.5,
                f"Latitude: {lat1}° to {lat2}°\n"
                f"Longitude: {lon1}° to {lon2}°\n"
                f"Peak Altitude AUGO1: {peak_alt_augo1}\n"
                f"Peak Altitude AUGSO: {peak_alt_augso}\n",
                fontsize=16, color="black", ha="left", va="center")
    # Save the figure only if save_condition is True
    if save_path and save_condition:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Figure saved at: {save_path}")
    plt.show()

    # Store results
#    results = {
#        "AUGO1": {
#            "peak_altitude": peak_alt_augo1,
#            "peak_rayleigh": peak_rayleigh_augo1,
#        },
#        "AUGSO": {
#            "peak_altitude": peak_alt_augso,
#            "peak_rayleigh": peak_rayleigh_augso,
#        },
#    }
    print(f"Peak Altitude AUGO1: {peak_alt_augo1}")
    print(f"Peak Rayleigh AUGO1: {peak_rayleigh_augo1}")
    print(f"Peak Altitude AUGSO: {peak_alt_augso}")
    print(f"Peak Rayleigh AUGSO: {peak_rayleigh_augso}")
#    return results

def calculation_for_conditions(rayleigh_augo1, rayleigh_augso, alt_fieldline):
    """
    Parameters:
        rayleigh_augo1 (list): list of Rayleigh values for AUGO1.
        rayleigh_augso (list): list of Rayleigh values for AUGSO.
        alt_fieldline (list): list of altitudes for each point.

    Returns:
        dict: Dictionary containing correlation, peak altitude, peak Rayleigh, centre, percentile ratio,
              mean Rayleigh above peak altitude, mean Rayleigh below peak altitude for both graphs.
    """
    # Remove NaN values for correlation computation
    valid_indices = ~np.isnan(rayleigh_augo1) & ~np.isnan(rayleigh_augso)
    correlation = np.corrcoef(rayleigh_augo1[valid_indices], rayleigh_augso[valid_indices])[0, 1]

    poly_func_augo1, peak_alt_augo1, peak_rayleigh_augo1 = fit_and_find_peak(alt_fieldline, rayleigh_augo1)
    poly_func_augso, peak_alt_augso, peak_rayleigh_augso = fit_and_find_peak(alt_fieldline, rayleigh_augso)

    # Compute Centre (Σ(Rayleigh * altitude) / ΣRayleigh)
    def compute_centre(altitudes, rayleigh_values):
        valid_idx = ~np.isnan(rayleigh_values)
        altitudes, rayleigh_values = altitudes[valid_idx], rayleigh_values[valid_idx]
        return np.sum(rayleigh_values * altitudes) / np.sum(rayleigh_values) if np.sum(rayleigh_values) != 0 else np.nan

    centre_augo1 = compute_centre(alt_fieldline, rayleigh_augo1)
    centre_augso = compute_centre(alt_fieldline, rayleigh_augso)

    # Compute P (60% percentile / median)
    def compute_p(rayleigh_values):
        valid_values = rayleigh_values[~np.isnan(rayleigh_values)]
        if len(valid_values) == 0:
            return np.nan
        percentile_60 = np.percentile(valid_values, 60)
        median_rayleigh = np.median(valid_values)
        return percentile_60 / median_rayleigh if median_rayleigh != 0 else np.nan

    p_augo1 = compute_p(rayleigh_augo1)
    p_augso = compute_p(rayleigh_augso)

    # Compute mean Rayleigh above and below peak altitude
    def mean_above_below_peak(altitudes, rayleigh_values, peak_altitude):
        valid_idx = ~np.isnan(rayleigh_values)
        altitudes, rayleigh_values = altitudes[valid_idx], rayleigh_values[valid_idx]

        above = rayleigh_values[altitudes > peak_altitude]
        below = rayleigh_values[altitudes < peak_altitude]

        mean_above = np.mean(above) if len(above) > 0 else np.nan
        mean_below = np.mean(below) if len(below) > 0 else np.nan

        return mean_above, mean_below

    mean_above_augo1, mean_below_augo1 = mean_above_below_peak(alt_fieldline, rayleigh_augo1, peak_alt_augo1)
    mean_above_augso, mean_below_augso = mean_above_below_peak(alt_fieldline, rayleigh_augso, peak_alt_augso)

    # Store results
    #print(f"Correlation: {correlation}")
    #print(f"Peak Altitude AUGO1: {peak_alt_augo1}")
    #print(f"Peak Rayleigh AUGO1: {peak_rayleigh_augo1}")
    #print(f"Centre AUGO1: {centre_augo1}")
    #print(f"P AUGO1: {p_augo1}")
    #print(f"Mean Rayleigh Above Peak AUGO1: {mean_above_augo1}")
    #print(f"Mean Rayleigh Below Peak AUGO1: {mean_below_augo1}")
    #print(f"Peak Altitude AUGSO: {peak_alt_augso}")
    #print(f"Peak Rayleigh AUGSO: {peak_rayleigh_augso}")
    #print(f"Centre AUGSO: {centre_augso}")
    #print(f"P AUGSO: {p_augso}")
    #print(f"Mean Rayleigh Above Peak AUGSO: {mean_above_augso}")
    #print(f"Mean Rayleigh Below Peak AUGSO: {mean_below_augso}")

    return correlation, peak_alt_augo1, peak_rayleigh_augo1, centre_augo1, p_augo1, mean_above_augo1, mean_below_augo1, peak_alt_augso, peak_rayleigh_augso, centre_augso, p_augso, mean_above_augso, mean_below_augso
def monte_carlo_peak_error_with_se(altitudes, rayleigh_values, rayleigh_std, sample_size, num_simulations=1000):
    """
    Function to find the error range of a peak using the Monte Carlo method (considering standard error).
    Parameters:.
        altitudes (np.array): An array of altitudes.
        rayleigh_values (np.array): array of Rayleigh values
        rayleigh_std (np.array): array of standard deviations of Rayleigh values at each altitude.
        sample_size (int): number of samples at each altitude
        num_simulations (int): number of simulations.
    Returns: An array of
        tuple: (mean of peak altitude, standard error of peak altitude, mean of peak Rayleigh values, standard error of peak Rayleigh values)
    """

    peak_altitudes = []
    peak_rayleighs = []

    valid_idx = ~np.isnan(rayleigh_values)
    altitudes = altitudes[valid_idx]
    rayleigh_values = rayleigh_values[valid_idx]
    rayleigh_std = rayleigh_std[valid_idx]

    if len(altitudes) == 0:
        return np.nan, np.nan, np.nan, np.nan

    # Calculate standard error
    rayleigh_se = rayleigh_std / np.sqrt(sample_size)

    for _ in range(num_simulations):
        noisy_rayleigh = rayleigh_values + np.random.normal(0, rayleigh_se, size=len(rayleigh_values))
        smoothed_rayleigh = medfilt(noisy_rayleigh, kernel_size=3)
        peak_idx = np.argmax(smoothed_rayleigh)
        peak_altitudes.append(altitudes[peak_idx])
        peak_rayleighs.append(smoothed_rayleigh[peak_idx])

    return (np.mean(peak_altitudes), np.std(peak_altitudes), np.mean(peak_rayleighs), np.std(peak_rayleighs))

# --- 1) 追加：下からトレース関数 -----------------------------------------
def field_line_tracing_bottomup(lat_fl, lon_fl, alt_fl, upper, lower, interval):  # NEW
    """
    (lat_fl, lon_fl, alt_bottom) から上方向だけをトレースする版
    """
    lat_fieldline = []
    lon_fieldline = []
    alt_fieldline = []

    alt_bottom = alt_fl - lower
    alt_top    = alt_fl + upper
    alt_range  = np.arange(alt_bottom, alt_top + 1e-6, interval)

    # 開始点は alt_bottom
    r_geo = (rr + alt_bottom) / rr

    lat_rad = np.deg2rad(lat_fl)
    lon_rad = np.deg2rad(lon_fl)

    x_geo = r_geo * np.cos(lat_rad) * np.cos(lon_rad)
    y_geo = r_geo * np.cos(lat_rad) * np.sin(lon_rad)
    z_geo = r_geo * np.sin(lat_rad)

    x_gsm, y_gsm, z_gsm = gp.Coords.GEOtoGSM(x_geo, y_geo, z_geo, dates, ut_time)

    for alt in alt_range:
        trace = gp.TraceField(x_gsm, y_gsm, z_gsm, dates, ut_time, coord_In='GSM', alt=alt)
        lat_fieldline.append(trace.GlatN)
        lon_fieldline.append(trace.GlonN)
        alt_fieldline.append(alt)

    # 上下で向きをそろえる（低→高に昇順）
    return lat_fieldline, lon_fieldline, alt_fieldline  # NEW
# --------------------------------------------------------------------------


# --- 2) 追加：top-down / bottom-up を別々に並べて描くプロット -------------
def plot_profiles_top_bottom(
    td_profiles_augo1, td_profiles_augso,
    bu_profiles_augo1, bu_profiles_augso,
    alt_fieldline, colors_td, labels_td, colors_bu, labels_bu,
    save_path=None
):
    import numpy as np
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)

    def _one(ax, profiles, colors, labels, title):
        arr = np.vstack(profiles) if profiles else np.empty((0, len(alt_fieldline)))
        mean = np.nanmean(arr, axis=0) if arr.size else np.full_like(alt_fieldline, np.nan, float)

        for prof, c, lab in zip(profiles, colors, labels):
            ax.plot(prof, alt_fieldline, lw=1.2, alpha=0.9, color=c, label=lab)

        ax.plot(mean, alt_fieldline, '-', lw=3.0, color='black', label='Mean')
        ax.set_title(title)
        ax.set_xlabel("Rayleigh")
        ax.grid(True)

        # ★ 凡例を外側へ
        ax.legend(
            fontsize=8,
            loc='upper left',
            bbox_to_anchor=(1.02, 1.0),    # ← パネルの右に配置
            borderaxespad=0.
        )

        return mean

    mean_td_augo1 = _one(axes[0,0], td_profiles_augo1, colors_td, labels_td, "Top-down (AUGO1)")
    mean_td_augso = _one(axes[0,1], td_profiles_augso, colors_td, labels_td, "Top-down (AUGSO)")
    mean_bu_augo1 = _one(axes[1,0], bu_profiles_augo1, colors_bu, labels_bu, "Bottom-up (AUGO1)")
    mean_bu_augso = _one(axes[1,1], bu_profiles_augso, colors_bu, labels_bu, "Bottom-up (AUGSO)")

    axes[0,0].set_ylabel("Altitude (km)")
    axes[1,0].set_ylabel("Altitude (km)")

    plt.tight_layout(rect=[0, 0, 0.85, 1])  # ← 右の凡例分のスペースを確保

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()

    return mean_td_augo1, mean_td_augso, mean_bu_augo1, mean_bu_augso
# --------------------------------------------------------------------------

def compute_peak_width(altitudes, rayleigh_values, frac):
    """
    altitudes:  高度配列（昇順）
    rayleigh_values: Rayleigh 配列（同じ長さ、NaN含み可）
    frac: ピーク値の何 % で幅を測るか（0.9 なら 90% 幅）

    Returns:
        width_km : 幅 [km]（high - low）
        low_alt  : 閾値を下回り始める高度
        high_alt : 閾値を下回り始める高度（上側）
    """
    altitudes = np.asarray(altitudes)
    rayleigh_values = np.asarray(rayleigh_values)

    # 有効データだけ抜き出し
    valid = np.isfinite(rayleigh_values)
    if valid.sum() < 3:
        return np.inf, np.nan, np.nan  # 幅条件ではじくために ∞ を返す

    alts = altitudes[valid]
    vals = rayleigh_values[valid]

    # 少し滑らかにする（fit_and_find_peak_med と同じノリ）
    smooth = medfilt(vals, kernel_size=3)

    # ピーク
    peak_idx = np.argmax(smooth)
    peak_alt = alts[peak_idx]
    peak_val = smooth[peak_idx]
    if peak_val <= 0:
        return np.inf, np.nan, np.nan

    threshold = peak_val * frac

    # 下側方向にたどる
    i_low = peak_idx
    while i_low > 0 and smooth[i_low] >= threshold:
        i_low -= 1
    low_alt = alts[i_low]

    # 上側方向にたどる
    i_high = peak_idx
    n = len(alts) - 1
    while i_high < n and smooth[i_high] >= threshold:
        i_high += 1
    high_alt = alts[i_high]

    width_km = high_alt - low_alt
    return width_km, low_alt, high_alt

from scipy.signal import medfilt

# ------------------------------------------------------------
# 1) 1リージョン分だけを処理するヘルパー
# ------------------------------------------------------------
def trace_and_filter_single_region(
    lat_range, lon_range, alt_fl,
    step, upper, lower, interval,
    peak, corr_thr, edge, frac, width_limit, median_thr,
    make_plots=True,
    save_prefix=None
):
    """
    1つの (lat_range, lon_range) に対してフィールドラインをトレースし，
    条件を満たしたフィールドラインの情報を返す。
    """

    lat_fl_values = np.arange(*lat_range, step)
    lon_fl_values = np.arange(*lon_range, step)

    # 条件を満たしたラインのプロファイル・座標を溜める
    all_rayleigh_augo1 = []
    all_rayleigh_augso = []
    lines_augo1 = []
    lines_augso = []

    alt_fieldline_last = None
    sample_size = 0

    for lat_fl in lat_fl_values:
        for lon_fl in lon_fl_values:

            # --- field-line tracing（上から下へ） ------------------------
            lat_td, lon_td, alt_td = field_line_tracing(
                lat_fl, lon_fl, alt_fl, upper, lower, interval
            )

            x1_td, y1_td = inverse_mapping(
                lat_td, lon_td, alt_td,
                lat0_augo1, lon0_augo1, h1, lat_size, lon_size
            )
            x2_td, y2_td = inverse_mapping(
                lat_td, lon_td, alt_td,
                lat0_augso,  lon0_augso,  h1, lat_size, lon_size
            )

            r1_td, r2_td, alt_td = rayleigh_vs_altitude(
                x1_td, y1_td, x2_td, y2_td, alt_td
            )

            # プロファイルの特徴量などを算出
            corr_td, palt1_td, pray1_td, cen1_td, p1_td, ma1_td, mb1_td, \
                palt2_td, pray2_td, cen2_td, p2_td, ma2_td, mb2_td = \
                calculation_for_conditions(r1_td, r2_td, alt_td)

            # 半値幅（90%幅）を計算
            width90_augo1_td, low90_augo1_td, high90_augo1_td = \
                compute_peak_width(alt_td, r1_td, frac=frac)
            width90_augso_td, low90_augso_td, high90_augso_td = \
                compute_peak_width(alt_td, r2_td, frac=frac)

            # --- 採択条件 ------------------------------------------------
            cond = (
                abs(palt1_td - palt2_td) < peak and
                alt_fl - lower + edge < palt1_td < alt_fl + upper - edge and
                alt_fl - lower + edge < palt2_td < alt_fl + upper - edge and
                abs(cen1_td - cen2_td) < median_thr and
                corr_td > corr_thr and
                (pray1_td > ma1_td * p1_td or pray1_td > mb1_td * p1_td) and
                (pray2_td > ma2_td * p2_td or pray2_td > mb2_td * p2_td) and
                width90_augo1_td <= width_limit and
                width90_augso_td <= width_limit
            )

            if cond:
                all_rayleigh_augo1.append(np.array(r1_td))
                all_rayleigh_augso.append(np.array(r2_td))

                lines_augo1.append(
                    (np.asarray(x1_td).ravel(), np.asarray(y1_td).ravel())
                )
                lines_augso.append(
                    (np.asarray(x2_td).ravel(), np.asarray(y2_td).ravel())
                )

                alt_fieldline_last = np.array(alt_td)
                sample_size += 1

    if sample_size == 0 or alt_fieldline_last is None:
        print(f"[{lat_range}, {lon_range}] No valid field lines.")
        return [], [], np.array([]), [], []

    # =================================================================
    # ここから「このリージョン単独の図」を描くパート
    # =================================================================
    if make_plots:
        label_str = f"lat={lat_range[0]}–{lat_range[1]}, lon={lon_range[0]}–{lon_range[1]}"

        # ----------------- (A) 全天画像 + フィールドライン -----------------
        fig_im, (ax_im1, ax_im2) = plt.subplots(1, 2, figsize=(12, 6))
        flipped_augo1 = cropped_augo1_np[:, ::-1]
        flipped_augso = cropped_augso_np[:, ::-1]
        ax_im1.imshow(flipped_augo1, cmap='gray')
        ax_im2.imshow(flipped_augso, cmap='gray')
        ax_im1.set_title(f"AUGO1: {label_str}")
        ax_im2.set_title(f"AUGSO: {label_str}")

        # 色のサイクル
        colors_cycle = [
            "#ff0000","#ff4000","#ff8000","#ffbf00","#ffff00",
            "#bfff00","#80ff00","#40ff00","#00ff80","#00ffbf",
            "#00ffff","#00bfff","#0080ff","#0040ff","#0000ff"
        ]
        line_colors = []   # ← CHANGED: 各ラインの色を記録

        for idx, ((x1, y1), (x2, y2)) in enumerate(zip(lines_augo1, lines_augso)):
            c = colors_cycle[idx % len(colors_cycle)]
            line_colors.append(c)   # ← CHANGED
            ax_im1.scatter(cropped_augo1_np.shape[1] - x1, y1, c=c, s=2,
                           label=f"line {idx+1}" if idx < len(colors_cycle) else None)
            ax_im2.scatter(cropped_augso_np.shape[1] - x2, y2, c=c, s=2,
                           label=f"line {idx+1}" if idx < len(colors_cycle) else None)

        # 凡例（重複除去）
        def uniq_legend(ax):
            h, l = ax.get_legend_handles_labels()
            if not h:
                return
            uniq = dict(zip(l, h))
            ax.legend(uniq.values(), uniq.keys(),
                      loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=8)

        uniq_legend(ax_im1)
        uniq_legend(ax_im2)

        plt.tight_layout()
        if save_prefix is not None:
            path_im = os.path.join(save_dir, f"{save_prefix}_fieldlines_single.png")
            plt.savefig(path_im, bbox_inches="tight")
            print(f"Saved: {path_im}")
        plt.show()

        # ----------------- (B) Rayleigh–Altitude プロファイル ---------------
        arr1 = np.vstack(all_rayleigh_augo1)  # (n_line, n_alt)
        arr2 = np.vstack(all_rayleigh_augso)

        mean1 = np.nanmean(arr1, axis=0)
        mean2 = np.nanmean(arr2, axis=0)

        mean1_s = medfilt(mean1, kernel_size=3)
        mean2_s = medfilt(mean2, kernel_size=3)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        alt = alt_fieldline_last

        # ------------------------------------------------------------
        # ★ Peak detection（平均プロファイルのピークを求める）
        # ------------------------------------------------------------
        idx_pk1 = np.nanargmax(mean1_s)    # ←★追加
        pk_ray1 = mean1_s[idx_pk1]         # ←★追加
        pk_alt1 = alt[idx_pk1]             # ←★追加

        idx_pk2 = np.nanargmax(mean2_s)    # ←★追加
        pk_ray2 = mean2_s[idx_pk2]         # ←★追加
        pk_alt2 = alt[idx_pk2]             # ←★追加

        print(f"{label_str} AUGO1:{pk_alt1}, AUGSO:{pk_alt2}")

        # AUGO1: 各ラインをカラーで表示（±1σなし）  # ← CHANGED
        for prof, c in zip(all_rayleigh_augo1, line_colors):
            ax1.plot(prof, alt, color=c, alpha=0.8, lw=1.2)
        # 黒太線で平均（スムージング済み）
        ax1.plot(mean1_s, alt, color='black', lw=2.5, label='Mean (smoothed)')
        ax1.set_xlabel("Rayleigh")
        ax1.set_ylabel("Altitude (km)")
        ax1.set_title(f"AUGO1 {label_str}")
        ax1.grid(True)
        ax1.legend(loc='best')

        # AUGSO: 同じ色対応で表示  # ← CHANGED
        for prof, c in zip(all_rayleigh_augso, line_colors):
            ax2.plot(prof, alt, color=c, alpha=0.8, lw=1.2)
        ax2.plot(mean2_s, alt, color='black', lw=2.5, label='Mean (smoothed)')
        ax2.set_xlabel("Rayleigh")
        ax2.set_title(f"AUGSO {label_str}")
        ax2.grid(True)
        ax2.legend(loc='best')

        plt.tight_layout()
        if save_prefix is not None:
            path_prof = os.path.join(save_dir, f"{save_prefix}_profiles_single.png")
            plt.savefig(path_prof, bbox_inches="tight")
            print(f"Saved: {path_prof}")
        plt.show()

    # ---- データを返す（multi-region 用でも使える） -------------------
    return all_rayleigh_augo1, all_rayleigh_augso, alt_fieldline_last, lines_augo1, lines_augso

# ------------------------------------------------------------
# 2) 複数リージョンをまとめて処理 & 図を描くメイン関数
#    （trace_and_filter_field_lines をこの形に差し替え）
# ------------------------------------------------------------
def trace_and_filter_field_lines(
    regions, colors,
    alt_fl, step, upper, lower, interval,
    peak, corr_thr, edge, frac, width_limit, median_thr,
    save_prefix=None
):
    """
    regions : list of dict
        例:
        regions = [
            {'lat1':52, 'lat2':54, 'lon1':-114, 'lon2':-112, 'label':'R1'},
            {'lat1':52, 'lat2':54, 'lon1':-113, 'lon2':-111, 'label':'R2'},
            {'lat1':52, 'lat2':54, 'lon1':-112, 'lon2':-110, 'label':'R3'}
        ]
    colors  : list of str
        各 region に対応する色（'red', 'limegreen', ...）

    役割：
      - すべてのリージョンで Single-region トレースを実行
      - 全天画像に R1–R3 のフィールドラインを色分け表示
      - 横長の Rayleigh–Altitude 図に R1–R3 の平均±1σ を表示
    """

    # ---- すべてのリージョンの結果を保存する辞書 ----
    region_data = {}
    common_alt = None  # 全リージョンで共通の高度配列（前提）

    for reg, col in zip(regions, colors):
        lat_range = (reg['lat1'], reg['lat2'])
        lon_range = (reg['lon1'], reg['lon2'])

        profs1, profs2, alt_arr, lines1, lines2 = trace_and_filter_single_region(
            lat_range, lon_range, alt_fl,
            step, upper, lower, interval,
            peak, corr_thr, edge, frac, width_limit, median_thr,
            make_plots=True, save_prefix=save_prefix
        )

        if len(profs1) == 0 or alt_arr.size == 0:
            continue

        if common_alt is None:
            common_alt = alt_arr
        else:
            # alt の長さが違うと扱いづらいので，違っていたら警告してスキップ
            if len(common_alt) != len(alt_arr) or not np.allclose(common_alt, alt_arr):
                print(f"Warning: altitude grid mismatch in {reg['label']}; skip.")
                continue

        region_data[reg['label']] = {
            "color": col,
            "alt": alt_arr,
            "profs_augo1": [np.array(p) for p in profs1],
            "profs_augso": [np.array(p) for p in profs2],
            "lines_augo1": lines1,
            "lines_augso": lines2,
        }

    if not region_data:
        print("No region produced valid field lines.")
        return

    # ========================================================
    #  (1) 全天画像にフィールドラインを色分け表示
    # ========================================================
    fig_im, (ax_im1, ax_im2) = plt.subplots(1, 2, figsize=(12, 6))
    flipped_augo1 = cropped_augo1_np[:, ::-1]
    flipped_augso = cropped_augso_np[:, ::-1]
    ax_im1.imshow(flipped_augo1, cmap='gray')
    ax_im2.imshow(flipped_augso, cmap='gray')
    ax_im1.set_title("AUGO1: all regions", fontsize=20)
    ax_im2.set_title("AUGSO: all regions", fontsize=20)

    for label, data in region_data.items():
        col = data["color"]

        # AUGO1
        for (x_arr, y_arr) in data["lines_augo1"]:
            ax_im1.scatter(
                cropped_augo1_np.shape[1] - x_arr, y_arr,
                c=col, s=2, label=label
            )
        # AUGSO
        for (x_arr, y_arr) in data["lines_augso"]:
            ax_im2.scatter(
                cropped_augso_np.shape[1] - x_arr, y_arr,
                c=col, s=2, label=label
            )

    # --- ISS Footprintなどの点群を全天画像上に描画 ---
    ax_im1.scatter(xy_augo1[:, 0], xy_augo1[:, 1], s=6, color='black', alpha=0.7, marker='o', label='Footprint')
    ax_im2.scatter(xy_augso[:, 0], xy_augso[:, 1], s=6, color='black', alpha=0.7, marker='o', label='Footprint')


    # 凡例は重複ラベルをまとめる
    def uniq_legend(ax):
        handles, labels = ax.get_legend_handles_labels()
        uniq = dict(zip(labels, handles))
        ax.legend(uniq.values(), uniq.keys(), loc='upper right', fontsize=15, framealpha=0.8)

    uniq_legend(ax_im1)
    uniq_legend(ax_im2)

    plt.tight_layout()
    if save_prefix is not None:
        path_im = os.path.join(save_dir, f"{save_prefix}_all_regions_fieldlines.png")
        plt.savefig(path_im, bbox_inches="tight")
        print(f"Saved: {path_im}")
    plt.show()

    # ========================================================
    #  (2) 横長の Rayleigh–Altitude 図（平均 ±1σ）
    # ========================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    for label, data in region_data.items():
        col = data["color"]
        alt = data["alt"]

        arr1 = np.vstack(data["profs_augo1"])  # (n_line, n_alt)
        arr2 = np.vstack(data["profs_augso"])

        mean1 = np.nanmean(arr1, axis=0)
        std1  = np.nanstd(arr1,  axis=0)
        mean2 = np.nanmean(arr2, axis=0)
        std2  = np.nanstd(arr2,  axis=0)

        # メディアンフィルタで平滑化（ピークを見やすく）
        mean1_s = medfilt(mean1, kernel_size=3)
        mean2_s = medfilt(mean2, kernel_size=3)

        # --- AUGO1 ---
        ax1.fill_betweenx(alt, mean1 - std1, mean1 + std1,
                          color=col, alpha=0.15)
        ax1.plot(mean1, alt, color=col, alpha=0.5, lw=1.0)
        ax1.plot(mean1_s, alt, color=col, lw=2.0, label=label)

        # --- AUGSO ---
        ax2.fill_betweenx(alt, mean2 - std2, mean2 + std2,
                          color=col, alpha=0.15)
        ax2.plot(mean2, alt, color=col, alpha=0.5, lw=1.0)
        ax2.plot(mean2_s, alt, color=col, lw=2.0, label=label)

    ax1.set_xlabel("Rayleigh", fontsize=20)
    ax1.set_ylabel("Altitude (km)", fontsize=20)
    ax1.set_title("Rayleigh vs Alt (AUGO1) — region means ±1σ", fontsize=20)
    ax1.grid(True)
    ax1.legend(loc="best")

    ax2.set_xlabel("Rayleigh", fontsize=14)
    ax2.set_title("Rayleigh vs Alt (AUGSO) — region means ±1σ", fontsize=20)
    ax2.grid(True)
    ax2.legend(loc="best")

    plt.tight_layout()
    if save_prefix is not None:
        path_prof = os.path.join(save_dir, f"{save_prefix}_region_means.png")
        plt.savefig(path_prof, bbox_inches="tight")
        print(f"Saved: {path_prof}")
    plt.show()
    return region_data


In [9]:
"""
いくつかの緯度経度範囲に対する
磁力線トレーシング法による高度プロファイル作成
（例）
緯度経度範囲
regions = [
    {'lat1':52, 'lat2':54, 'lon1':-111, 'lon2':-109, 'label':'R1'},
    {'lat1':53, 'lat2':54, 'lon1':-112.5, 'lon2':-111, 'label':'R2'},
    {'lat1':54, 'lat2':55, 'lon1':-112.5, 'lon2':-111, 'label':'R3'}
]

高度範囲：55km~150km 2kmおき（基準値alt_fl=100, 上upper=50km, 下lower=45km, interval=2km)
磁力線トレーシングの条件
二地点の高度プロファイルのピークの差：5km peak=5
二地点の高度プロファイルの相関係数 0.8以上 corr_thr=0.8
二地点それぞれのピークが高度範囲の上端下端10kmに入っていないか　edge=10
二地点それぞれのピークがしっかりとあり、90％幅40km以上　frac=0.9, width_limit=40
二地点の中央値の差20km以内 median_thr=20
region_data=trace_and_filter_field_lines(
    regions, colors,
    alt_fl=100, step=0.1,
    upper=50, lower=45, interval=2,
    peak=5, corr_thr=0.8, edge=10, width_limit=40,
    save_prefix=date_time
)

"""

regions = [
    {'lat1':52, 'lat2':54, 'lon1':-111, 'lon2':-109, 'label':'R1'},
    {'lat1':53, 'lat2':54, 'lon1':-112.5, 'lon2':-111, 'label':'R2'},
    {'lat1':54, 'lat2':55, 'lon1':-112.5, 'lon2':-111, 'label':'R3'}
]

colors = ['red', 'green', 'deepskyblue']


region_data=trace_and_filter_field_lines(
    regions, colors,
    alt_fl=100, step=0.1,
    upper=50, lower=45, interval=2,
    peak=5, corr_thr=0.8, edge=10, frac=0.9, width_limit=40, median_thr=20,
    save_prefix=date_time
)

Output hidden; open in https://colab.research.google.com to view.